<a href="https://colab.research.google.com/github/YJR66/-/blob/main/fine_tune_gemma_4_on_emotions_final_zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 使用 `dair-ai/emotion` 数据集微调 Gemma 4 E4B-it



**文章来源：**[如何微调 Gemma 4：基于人类情绪数据集的完整实战指南](https://www.datacamp.com/tutorial/fine-tune-gemma-4)

Colab 来源：[在 dair-ai/emotion 上微调 Gemma 4 E4B-it](https://huggingface.co/kingabzpro/gemma4-emotion-lora/blob/main/fine-tune-gemma-4-on-emotions_final.ipynb)

这个 Colab 笔记本的主要功能是**对 Google 的 Gemma 4 模型（具体为 `google/gemma-4-E4B-it`）进行微调，以实现人类情感分类任务**。

以下是该笔记本核心功能的详细分析：

### 1\. 任务目标与数据集

* **目标任务**：将输入的文本分类为六种情感之一：悲伤 (sadness)、快乐 (joy)、爱 (love)、愤怒 (anger)、恐惧 (fear) 或惊讶 (surprise)。  
* **数据集**：使用了 Hugging Face 上的 `dair-ai/emotion` 数据集。笔记本对数据进行了采样限制（如训练集 4000 条，测试集 400 条），以便在 Colab 环境下快速运行。

### 2\. 核心技术栈

* **基础模型**：`google/gemma-4-E4B-it`，这是一个指令微调版的轻量级大模型。  
* **优化技术**：  
  * **4-bit 量化**：使用 `bitsandbytes` 库以 4-bit 模式加载模型，极大地降低了显存占用。  
  * **LoRA 微调 (PEFT)**：通过 `peft` 库配置 LoRA（Low-Rank Adaptation），仅训练一小部分参数（约 52M 参数），而非全量微调，从而在有限算力下提升性能。  
  * **SFT 训练**：使用 `trl` 库中的 `SFTTrainer` 进行监督式微调。

### 3\. 完整的工作流程

笔记本按照标准的机器学习流程组织：

1. **环境准备**：安装 `transformers`、`peft`、`trl` 等核心库，并通过 `HF_TOKEN` 登录 Hugging Face。  
2. **数据处理**：将原始数据集转换为 TRL 推荐的“指令-回答”聊天格式（Chat Format），并加入系统提示词（System Prompt）来规范模型的输出。  
3. **基准评估**：在微调之前，先对原始的 Gemma 4 模型进行测试。结果显示其原始准确率约为 **58.25%**。  
4. **模型训练**：配置 LoRA 参数（如 rank=16）和训练参数（如学习率 1e-4），执行 1 个 Epoch 的训练。  
5. **对比评估**：微调后再次评估，准确率显著提升至约 **82.25%**，大幅改善了情感分类的准确性。  
6. **模型保存**：将训练好的 LoRA 适配器（Adapter）和分词器保存到本地，并尝试上传至 Hugging Face Hub。

### 4\. 评估指标

笔记本使用了 `scikit-learn` 提供的一系列指标来衡量性能变化，包括：

* **准确率 (Accuracy)**。  
* **F1 分数**。  
* **混淆矩阵 (Confusion Matrix)**：用于分析模型在哪些具体情感类别上容易产生误判。

**总结**：这是一个非常实用的教学/工程模板，演示了如何利用现代高效微调技术（QLoRA）将一个通用大语言模型转化为特定的分类专家。

笔记本中使用的 `google/gemma-4-E4B-it` 是 Google 最近发布的 Gemma 4 系列中的“高效能”版本。

针对这款模型，显存（VRAM）的占用情况主要取决于**加载方式**（是否量化）以及**上下文长度**：

### 1\. 显存占用详情 (VRAM)

根据代码（使用了 `load_in_4bit=True`），配置属于 **4-bit 量化加载**。

| 加载模式 | 静态模型占用 (模型权重) | 推荐 GPU 显存 (建议预留) | 备注 |
| :---- | :---- | :---- | :---- |
| **4-bit 量化 (配置)** | **\~5.5 GB** | **8 GB \- 10 GB** | 适合大多数家用显卡 (如 RTX 3060/4060) |
| **8-bit 量化** | \~9.5 GB | 12 GB \- 16 GB | 精度更高，显存需求翻倍 |
| **BF16/FP16 (全精度)** | \~15 GB | 20 GB \- 24 GB | 适合 A100/H100 或 RTX 3090/4090 |

### 2\. 为什么代码能跑通？

使用的是 **4-bit 量化**，模型权重本身只占约 **5.5GB**。由于是在 Colab 上运行，通常分配的是 **T4 (16GB)** 或 **L4 (24GB)** GPU。

* **模型权重**：\~5.5GB  
* **LoRA 适配器 (代码里有)**：约几十 MB，几乎不占显存。  
* **训练缓存/梯度**：在训练时会产生额外开销。  
* **总占用**：微调任务中，总显存占用大约在 **8GB 到 10GB** 之间。

### 3\. 一个需要注意的“陷阱”：上下文窗口

Gemma 4 E4B 支持高达 **128K** 的上下文窗口。

* **警告**：虽然模型权重本身很小，但如果输入的文本非常长（例如一次性丢入几万字），**KV Cache (键值缓存)** 会消耗巨大的显存，甚至可能超过模型本身的大小。  
* **建议**：在微调时，保持 `max_seq_length` 在合理范围（如 512, 1024 或 2048），这样 16GB 的显存就绰绰有余。

**总结：** 4-bit 配置在拥有 **10GB 以上显存** 的 GPU 上就能跑得很稳。Colab 的默认显存完全能够胜任。

如果是进行 **Training（微调/训练）**，显存的需求会比 **Inference（推理）** 高出很多，通常是 **2 倍甚至更多**。

用的 `google/gemma-4-E4B-it`（总参数约 8B，含 Embedding），在 **Training** 模式下的显存占用主要由以下四个部分组成：

### 1\. 显存占用四大金刚（训练维度）

| 组成部分 | 推理 (Inference) | 训练 (Training \- QLoRA 模式) |
| :---- | :---- | :---- |
| **模型权重** | \~5 GB (4-bit) | \~5 GB (4-bit 固定权重) |
| **优化器状态** | 无 | **\~1-2 GB** (LoRA  trainable params) |
| **梯度 (Gradients)** | 无 | **\~1 GB** (需要保存梯度进行更新) |
| **激活值 (Activations)** | 极小 | **2 GB \- 10 GB+** (取决于 Batch Size 和 Seq Length) |

### 2\. 实测估算：Gemma 4 E4B (QLoRA 微调)

Colab 脚本中（LoRA Rank=16, 4-bit 量化），显存开销大致如下：

* **基础开销（模型 \+ 系统）**：约 **6-7 GB**。  
* **训练动态开销**：  
  * **Max Sequence Length \= 512**: 总显存约需 **10-12 GB**。  
  * **Max Sequence Length \= 1024**: 总显存约需 **14-16 GB**。  
  * **开启 Gradient Checkpointing**：可以显著降低激活值占用的显存，让 16GB 的 GPU 跑得更稳。

### 3\. 硬件建议

根据训练任务，建议如下：

* **入门级 (Colab T4 / RTX 3060 12G/4070)**：  
    
  * **能跑吗？** 能，但必须开启 `gradient_checkpointing=True` 且 `batch_size` 设为 1 或 2。  
  * **限制**：序列长度（Seq Length）不能太长，否则会 **OOM (Out of Memory)**。


* **进阶级 (Colab L4 24G / RTX 3090/4090 24G)**：  
    
  * **能跑吗？** 非常轻松。  
  * **优势**：可以支持更大的 Batch Size（如 4 或 8），训练速度会快 2-3 倍，且能处理 2048 以上的长文本。


* **生产级 (A100 40G/80G)**：  
    
  * 可以尝试 **全量微调 (Full Fine-tuning)** 而不仅是 LoRA，或者处理极长文本。

### 💡 训练调优小技巧 (省显存)

1. **`gradient_checkpointing=True`**：在 `SFTConfig` 或 `TrainingArguments` 中开启，能用稍微降低速度的代价换取巨大的显存空间。  
2. **`fp16` 或 `bf16`**：代码中使用了 `bf16=True`，这在 Ampere 架构（如 L4, A100）上非常高效且省空间。  
3. **`gradient_accumulation_steps`**：如果显存不够大，把 `per_device_train_batch_size` 设为 1，然后增加 `gradient_accumulation_steps`（比如设为 4），效果等同于 Batch Size 为 4。

**总结：** 针对 Gemma 4 E4B，**16GB 显存是“及格线”**（刚好够用），**24GB 显存是“舒适区”**（可以随心所欲调参）。  

在 Hugging Face 或 Google 的模型命名规则中，`it` 是 **Instruction Tuned**（指令微调）的缩写。

具体来说，它代表了模型在预训练（Pre-training）的基础上，又经过了专门的训练步骤，使其能够更好地**理解并执行人类的指令**，并以**对话的形式**进行交互。

以下是 `it` 版模型与基础版（Base/PT）模型的主要区别：

---

### 1. 核心定义
* **Base 模型 (预训练版):** 仅在海量文本上进行了“接龙”训练。如果问它“如何制作蛋糕？”，它可能不会回答步骤，而是列出一堆蛋糕店的名字，因为它认为自己在填充网页内容。
* **it 模型 (指令微调版):** 在 Base 模型基础上，使用了 **SFT** (监督微调) 和 **RLHF** (基于人类反馈的强化学习) 技术。它知道提出问题时，它的任务是提供准确、有逻辑的回答。

### 2. 训练目标的差异


| 特性 | Base 模型 (无 `it` 后缀) | Instruction Tuned (`it` 后缀) |
| :--- | :--- | :--- |
| **主要用途** | 进一步微调、少样本学习 (Few-shot) | **直接作为聊天机器人、助手使用** |
| **交互方式** | 文本补全 (Autocomplete) | **对话交互 (Q&A / Chat)** |
| **安全性** | 较少经过安全对齐，可能输出有害内容 | 经过安全护栏训练，会拒绝不当请求 |
| **提示词要求** | 需要构造复杂的上下文 | 直接使用自然语言指令即可 |

### 3. 为什么选择 `it` 版本？
如果正在开发一个**问答系统、代码助手或聊天机器人**，应该始终优先选择带有 `-it` 后缀的模型。因为这些模型已经学习了特定的对话模板（如控制何时结束对话、如何区分用户指令和模型回复）。

## 1. 安装依赖

## 核心库安装与环境配置

该单元格通过 `pip` 安装并更新了微调 **Gemma ４**（或其它大语言模型）所必需的核心组件。

### 单元格指令
* **`%%capture`**: 这是一个 Colab/Jupyter 的魔术命令，用于捕获并隐藏该单元格的所有输出信息（如安装过程中的进度条和日志），使笔记本界面保持整洁。

### 主要依赖库
| 库名称 | 功能描述 |
| :--- | :--- |
| **`transformers`** | Hugging Face 的核心库，用于加载、处理和训练各种预训练模型。 |
| **`accelerate`** | 帮助在各种设备（如单机多卡、TPU）上轻松运行 PyTorch 代码，优化并行计算。 |
| **`datasets`** | 用于轻松下载和处理 Hugging Face Hub 上的各种 NLP 数据集（本例中为 `dair-ai/emotion`）。 |
| **`trl`** | *Transformer Reinforcement Learning*，专门用于训练、微调和对齐语言模型的库，提供了 **SFT**（有监督微调）的工具。 |
| **`peft`** | *Parameter-Efficient Fine-Tuning*，提供轻量化微调技术（如 **LoRA**），允许在不训练全部参数的情况下实现高效微调，大幅降低显存需求。 |
| **`bitsandbytes`** | 用于模型**量化**（如 4-bit 或 8-bit 加载），是低显存环境下运行大模型的关键。 |
| **`scikit-learn`** | 经典的机器学习库，用于计算评估指标（如准确率、F1 分数）和生成混淆矩阵。 |
| **`huggingface_hub`** | 允许程序与 Hugging Face 平台交互，进行模型上传、下载及登录操作。 |

In [ ]:
%%capture
!pip install -U transformers accelerate datasets trl peft bitsandbytes scikit-learn huggingface_hub

## 2. 使用 `HF_TOKEN` 登录

这段代码是微调大模型（LLM）的第一步：**身份认证与安全接入**。在 Colab 环境下，它建立了一个安全的通道，能够合法地从 Hugging Face 下载受限模型（如 Gemma 4）并将微调后的结果上传回云端。

以下是代码的详细逻辑解析：

---

## 🔑 Hugging Face 身份验证代码解析

### 1. 核心模块导入
* **`from huggingface_hub import login`**: 导入 Hugging Face 官方提供的登录函数。它会将 Access Token 存储在本地配置文件中，后续所有调用 HF API 的操作（如 `from_pretrained`）都会自动带上这个身份证明。
* **`from google.colab import userdata`**: 这是 **Google Colab 环境特有的** 工具。它允许代码从 Colab 左侧侧边栏的“钥匙”图标（Secrets）中安全地读取环境变量，而无需直接在代码中硬编码敏感信息。

### 2. 安全读取机制 (`try...except`)
为了防止因为缺少 Token 导致程序崩溃或隐私泄露，代码采用了结构化的错误处理：

* **`userdata.get('HF_TOKEN')`**: 尝试从 Colab 的秘密库中提取名为 `HF_TOKEN` 的值。这是目前最推荐的做法，比直接在代码里写 `token="hf_xxx"` 要安全得多。
* **`login(token=hf_token)`**: 执行实际的登录动作。一旦成功，当前会话将获得访问私有库和受门控模型（Gated Models）的权限。

### 3. 错误处理与用户反馈
这段代码通过不同的异常捕获来提供“保姆级”提示：

| 错误类型 | 触发场景 | 反馈效果 |
| :--- | :--- | :--- |
| **`SecretNotFoundError`** | 忘记在 Colab 左侧“🔑”面板添加 `HF_TOKEN`。 | 抛出一个清晰的 `ValueError`，明确指引去哪里添加密钥。 |
| **`Exception`** | Token 过期、网络连接受阻或输入了错误的 Token。 | 打印具体的报错信息 `e`，方便快速排查网络或权限问题。 |

---

## 💡 为什么微调 Gemma 4 必须执行这一步？

1.  **门控模型协议 (Gated Models)**: Google 的 Gemma 系列模型通常需要 Hugging Face 页面上先接受使用协议。只有登录了拥有权限的账号，代码才能顺利下载权重文件。
2.  **模型推送 (Push to Hub)**: 如果在训练结束后，将专属微调模型通过 `model.push_to_hub("my-gemma-4-ft")` 保存起来，登录身份是唯一的写权限来源。
3.  **避免 API 限制**: 匿名下载大模型经常会遇到限速，而登录状态下的下载速度和稳定性更高。



**建议操作建议：**
在运行此代码前，请确保已经：
1. 在 [Hugging Face Settings](https://huggingface.co/settings/tokens) 创建了一个 **Write** 类型的 Token。
2. 在 Colab 左侧 **🔑 (Secrets)** 面板中，新增名称为 `HF_TOKEN` 的项，并将 Token 粘贴进去，同时勾选 **"Notebook access"**。

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print("Logged in to Hugging Face successfully.")
except userdata.SecretNotFoundError:
    raise ValueError("HF_TOKEN not found in Colab secrets. Please add it via the 🔑 panel.")
except Exception as e:
    print(f"An error occurred during login: {e}")

Logged in to Hugging Face successfully.


## 3. 加载数据集

这段代码标志着微调流程从“环境准备”进入了**“数据工程”**阶段。其核心目标是加载情感分类数据集，并根据算力资源对数据规模进行人工干预。

以下是代码的深度解析：

---

## 📊 数据集加载与规模管控解析

### 1. 规模预设 (Constants)
代码开头定义了一系列常量（如 `TRAIN_LIMIT = 4000`）。在微调 Gemma 4 这样的大模型时，这是一种非常务实的做法：
* **显存与时间平衡**：全量微调可能耗时过长，通过限制样本数（例如 4000 条训练数据），可以在几十分钟内完成验证，适合实验阶段。
* **代表性抽样**：对于简单的任务（如 6 分类情感分析），几千条高质量数据通常足以让 LoRA 适配器学到任务特征。

### 2. 数据源加载 (`load_dataset`)
* **`load_dataset("dair-ai/emotion")`**: 从 Hugging Face Hub 远程下载名为 `emotion` 的数据集。
* **数据集背景**：这是一个经典的情感分类数据集，包含 `sadness` (0), `joy` (1), `love` (2), `anger` (3), `fear` (4), `surprise` (5) 六个类别。

### 3. 数据截断逻辑 (`maybe_limit` 函数)
这是代码中最关键的逻辑组件，确保了数据的**随机性**和**安全性**：
* **`.shuffle(seed=42)`**: 在截取前先打乱顺序。`seed=42` 是“宇宙终极答案”，保证了实验的可重复性（每次运行选出的数据都一样）。
* **`.select(range(min(...)))`**:
    * 使用 `select` 而非 Python 切片，是因为 Hugging Face `datasets` 基于 Apache Arrow 内存映射，这样做不会直接消耗内存，速度极快。
    * `min(limit, len(split))` 是一个防错机制，防止要求的数量超过数据集本身的样本总量。



### 4. 结构化组织 (`DatasetDict`)
代码最后将处理后的数据重新封装为 `DatasetDict` 对象。这种结构的优势在于：
* **无缝衔接**：后续的 `SFTTrainer` (有监督微调训练器) 可以直接识别其中的 `train` 和 `validation` 字段。
* **保持对齐**：确保了训练、验证、测试三套数据集在预处理步骤上的一致性。

---

## 💡 为什么微调 Gemma 4 时需要限制数据量？

1.  **快速迭代**：在微调大模型时，调参（如学习率、LoRA 的 $r$ 值）比堆砌数据量更重要。先用 4000 条数据跑通闭环，比直接上万条数据要高效得多。
2.  **避免过拟合**：Gemma 4 拥有强大的推理能力。如果在一个极小领域的简单任务上使用过大的数据集进行多轮训练，模型可能会产生“灾难性遗忘”，失去原有的泛化能力。
3.  **计算成本**：即便使用了量化技术，每条数据的推理和反向传播依然需要计算。限制数量可以显著降低在 Colab 等环境下的 T4/L4 GPU 压力。

In [ ]:
from datasets import load_dataset, DatasetDict

TRAIN_LIMIT = 4000
VALIDATION_LIMIT = 400
TEST_LIMIT = 400
EVAL_LIMIT = 400

raw_dataset = load_dataset("dair-ai/emotion")

def maybe_limit(split, limit):
    split = split.shuffle(seed=42)
    if limit is None:
        return split
    return split.select(range(min(limit, len(split))))

dataset = DatasetDict({
    "train": maybe_limit(raw_dataset["train"], TRAIN_LIMIT),
    "validation": maybe_limit(raw_dataset["validation"], VALIDATION_LIMIT),
    "test": maybe_limit(raw_dataset["test"], TEST_LIMIT),
})

dataset


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 4000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 400
    })
})

这段代码虽然只有两行，但在模型微调中起到了**“建立语义映射”**的关键作用。它负责将枯燥的数字标签（0, 1, 2...）还原为人类可读的文字标签。

---

## 🏷️ 标签映射与语义解析

### 1. 代码逐层拆解
* **`dataset["train"]`**: 从 `DatasetDict` 中选定训练集部分。
* **`.features`**: 访问数据集的元数据（Metadata）结构。这就像查看数据库的表头定义，它记录了每一列的数据类型。
* **`["label"]`**: 准确定位到名为 `label` 的特征列。在 `dair-ai/emotion` 数据集中，这一列存储的是整数。
* **`.names`**: 这是最核心的属性。它提取出与这些整数索引相对应的**原始字符串列表**。

### 2. 输出结果预览
执行这段代码后，通常会得到如下列表：
```python
['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']
```

---

## 💡 为什么这一步在微调 Gemma 4 时至关重要？

### A. 配置模型的分类头 (Classification Head)
当微调 LLM 进行序列分类任务时，需要告诉模型：
1.  **分类数量 (num_labels)**：通过 `len(label_names)` 自动获取，避免硬编码。
2.  **ID 到文字的映射 (id2label)**：
    * `0 -> sadness`
    * `1 -> joy`
    * 这样当模型预测出结果后，可以直接看到“joy”而不是数字 1。

### B. 构建训练 Prompt
在微调 Gemma 4 这种生成式模型做分类时，我们通常会将标签融入 Prompt 中。例如：
> “这段文本的情感是：[label_name]”

有了这个 `label_names` 列表，可以在预处理阶段，通过索引轻松地将数据集中的数字 `0` 转换成单词 `sadness`，从而构建出高质量的训练指令。



### C. 评估与可视化
在训练完成后，会生成**混淆矩阵 (Confusion Matrix)**。如果没有这行代码提取出的名称，矩阵坐标轴上只会显示 0 到 5，而有了它，可以直观地看到模型是否经常把 `anger` 误判为 `fear`。

---

In [ ]:
label_names = dataset["train"].features["label"].names
label_names

['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']

这条代码执行的是**“样本抽样检查”**。在数据预处理的长跑中，这相当于停下来看一眼地图，确认数据的格式是否符合预期。

执行 `dataset["train"][0]` 会返回训练集中的**第一个样本对象**（通常是一个 Python 字典）。

---

## 🔍 数据样本结构深度解析

对于 `dair-ai/emotion` 数据集，输出结果通常如下所示：

```python
{
    'text': 'i didnt feel humiliated',
    'label': 0
}
```

### 1. 字段含义
* **`text`**: 原始的输入文本。这是 Gemma 4 将要读取并理解的内容。
* **`label`**: 情感类别的整数索引。根据上一行获取的 `label_names`，这里的 `0` 对应的是 `sadness`。

### 2. 为什么在微调 Gemma 4 前要看这一眼？

#### A. 验证清洗效果
在大模型微调中，**“垃圾进，垃圾出” (Garbage In, Garbage Out)** 是金科玉律。通过查看样本，可以确认：
* 文本是否包含乱码或异常字符。
* 文本长度是否符合预期（Gemma 4 虽然支持长文本，但如果数据全是超短句，可能需要调整策略）。

#### B. 规划 Prompt 模板
Gemma 4 是一个指令遵循（Instruction-tuned）模型。不能直接把上面的字典喂给它，需要根据这个结构设计 **Prompt 模板**。

例如，可能会根据这个样本构思出如下微调格式：
> **指令**: 分析下列文本的情感。
> **输入**: i didnt feel humiliated
> **答案**: sadness

#### C. 确定数据类型
通过这一步，确认了 `label` 是 `int` 类型而非 `string`。这意味着在后续配置 `AutoModelForSequenceClassification` 时，需要明确指定类别数量 `num_labels=6`。



---

### 🚀 进阶技巧：结合标签名称查看
如果想看得更直观，可以结合之前定义的 `label_names` 运行这段代码：

```python
sample = dataset["train"][0]
print(f"文本内容: {sample['text']}")
print(f"情感标签: {label_names[sample['label']]} (ID: {sample['label']})")
```

In [ ]:
dataset["train"][0]

{'text': 'while cycling in the country', 'label': 4}

## 4. 创建系统提示词

这段代码定义了微调过程中的**核心行为准则**，即 **System Prompt（系统提示词）**。它是给大模型（LLM）注入“角色灵魂”和“输出规范”的关键步骤。

对于像 **Gemma 4** 这样经过指令微调（Instruction-tuned）的模型，系统提示词能够显著提高模型在特定任务上的遵循度。

---

## 🛠️ System Prompt 的深度拆解

这段代码通过三个维度对模型进行了严格约束：

### 1. 角色设定 (Role Definition)
* **内容**: `"You are an emotion classification assistant."`
* **作用**: 将模型从一个“无所不知的通用对话者”切换为“专注的情感分类专家”。这有助于模型激活其参数中关于情感理解、文本分析相关的神经元区域。

### 2. 输出空间限制 (Output Space Constraint)
* **内容**: `"Only choose from: sadness, joy, love, anger, fear, surprise."`
* **作用**: 这是防止模型“幻觉”或“自由发挥”的关键。它明确规定了回答的边界，确保模型不会输出类似 "The person feels happy" 这样描述性的短语，而是严格输出预定义的标签。

### 3. 格式强制 (Formatting Rigidity)
* **内容**: `"Return only the label and nothing else."`
* **作用**: **非常重要！** 在自动化微调和批量评估时，我们只需要标签单词（如 `joy`）。如果模型返回 `"The emotion is joy."`，解析程序就会出错。这个指令极大地降低了数据后处理的难度。

---

## 💡 在微调 Gemma 4 中的战略意义

### A. 对齐训练目标
在进行 **SFT（有监督微调）** 时，我们会将这个 `SYSTEM_PROMPT` 作为输入的一部分。通过反复训练，模型会形成一种条件反射：一旦看到这段系统提示，它就知道接下来的任务是“六选一”的填空题。

### B. 适配 Gemma 的 Prompt 结构
Gemma 模型通常遵循特定的对话模板（如 `<start_of_turn>user` 等）。这段 `SYSTEM_PROMPT` 之后通常会拼接用户的 `text`，最终形成类似下面的训练样本：

> **System**: You are an emotion classification assistant...
> **User**: i didnt feel humiliated
> **Assistant**: sadness



### C. 零样本（Zero-shot）能力的基石
即使你还没开始正式训练，给 Gemma 4 加上这个系统提示词，它通常也能表现出不错的基础分类能力。微调的过程实际上是在强化这种“听话”的能力，并让它更精准地契合 `dair-ai/emotion` 这个数据集的分类习惯。

---

In [ ]:
SYSTEM_PROMPT = """You are an emotion classification assistant.
Read the user's text and answer with exactly one label.
Only choose from: sadness, joy, love, anger, fear, surprise.
Return only the label and nothing else."""

## 5. 将数据集转换为 TRL 的 prompt-completion 对话格式


这段代码是微调流程中极其关键的**数据清洗与格式化**步骤。它的目的是将原始的“纯文本+数字标签”结构，转换成大语言模型（LLM）能够理解的**对话式指令结构（Chat Template）**。

以下是代码的深度解析：

---

## 🛠️ 数据转换逻辑解析

### 1. `to_prompt_completion` 函数：构建对话模型
这个函数定义了数据如何被“喂”给 Gemma 4。它将一个简单的样本拆解为三个角色：

* **`system` (系统层)**：植入了之前定义的 `SYSTEM_PROMPT`。它告诉模型：“你是谁，你应该遵守什么规则”。
* **`user` (用户层)**：构造了具体的指令任务。通过 f-string 将原始文本嵌入到固定模板中：`"Classify the emotion of this text:\n\n{text}"`。这样做是为了让模型明确知道哪部分是指令，哪部分是待处理数据。
* **`assistant` (助手层/标签层)**：这是模型微调的**目标（Ground Truth）**。我们将数字标签通过 `label_names` 转换回了单词（如 `joy`）。模型在微调时，会学习在给定上述 `system` 和 `user` 内容的情况下，预测出这个 `assistant` 的内容。

### 2. `dataset.map`：全量应用与重构
* **`map(...)`**：这是 `datasets` 库的高效批处理方法，会将上面的函数应用到 `train`、`validation` 和 `test` 的每一条数据上。
* **`remove_columns=...`**：这是一个非常干净利落的操作。它删除了原始的 `text` 和 `label` 列，只保留新生成的 `prompt` 和 `completion`。
    * **为什么要删除？** 因为在 SFT（有监督微调）阶段，模型只需要结构化的对话数据，多余的原始列会占用显存并干扰 Data Collator。

---

## 💡 为什么这种格式对微调 Gemma 4 至关重要？

### A. 遵循 ChatML 标准
现代 LLM（尤其是 Gemma、Llama 系列）通常采用类似 ChatML 的格式进行预训练。将数据组织成 `role` 和 `content` 的列表，可以方便后续调用 `tokenizer.apply_chat_template`，自动插入模型特定的起始和结束标记（如 `<start_of_turn>`）。



### B. 区分“指令”与“数据”
如果不使用这种结构，模型可能会分不清哪些是你的要求，哪些是用户输入的干扰信息。明确的 `user` 角色标识符能显著提升模型在处理复杂文本时的分类稳定性。

### C. 零样向少样本平滑过渡
由于这种格式模拟了真实的对话场景，经过这样微调的模型，不仅在分类任务上表现更好，而且能保持更好的指令遵循能力，不会因为微调而变“傻”。

---

## 🔍 处理后的数据预览
现在，如果你查看 `sft_dataset["train"][0]`，结构会从原来的：
`{'text': '...', 'label': 0}`
**变成：**
```python
{
    "prompt": [
        {"role": "system", "content": "You are..."},
        {"role": "user", "content": "Classify... text"}
    ],
    "completion": [
        {"role": "assistant", "content": "sadness"}
    ]
}
```

**Gemma 4** 系列引入了原生的**“思维模式（Thinking Mode）”**，这标志着它在处理复杂逻辑和代理工作流（Agentic Workflows）方面有了质的飞跃。

以下是 Gemma 4 的各个版本及其对 Thinking 功能的支持情况：

### 1. Gemma 4 全系列版本
Gemma 4 放弃了以前简单的参数量命名方式，采用了针对不同硬件优化的规格：

| 版本名称 | 架构类型 | 核心定位 | 多模态支持 |
| :--- | :--- | :--- | :--- |
| **Gemma 4 E2B** | Effective 2B | 针对手机等移动设备优化，极致能效。 | 文本、图像、**音频** |
| **Gemma 4 E4B** | Effective 4B | 笔记本电脑、消费级显卡的黄金平衡版本。 | 文本、图像、**音频** |
| **Gemma 4 26B** | MoE (混合专家) | 高级推理能力，适合单张 80GB 显卡运行。 | 文本、图像 |
| **Gemma 4 31B** | Dense (稠密) | 旗舰版本，逻辑推理和编码能力对标顶尖商业模型。 | 文本、图像 |

---

### 2. Thinking 功能的应用规则
在 Gemma 4 中，**全系列模型都内置了推理引擎**，但它们的“Thinking”触发机制和表现略有不同：

#### **如何开启 Thinking 功能？**
Gemma 4 不再需要特殊的“Thinking 版”模型，而是通过**控制 Token** 切换：
* **触发开关**：在 `system` prompt 的开头加入 `<|think|>` 标记。
* **输出结构**：开启后，模型会按照以下结构输出：
    `<|channel>thought\n [内部推理过程] <channel|> [最终答案]`

#### **版本间的差异表现：**
* **E2B 和 E4B (移动端版本)**：
    这两个版本对 Thinking 模式做了物理级开关优化。如果你**不添加** `<|think|>` 标记，它们会直接跳过思考过程，以节省电量和减少首字延迟（TTFT）。
* **26B 和 31B (高性能版本)**：
    即便你**禁用了** Thinking 模式（不加标记），这些大参数模型在处理复杂逻辑时仍可能保留空的思考占位符，或者默认进行隐式的深层推理。

---

### 💡 针对你的微调需求建议
既然你使用的是 **E4B** 版本，这实际上是应用你“自定义推理流程”的最佳实验平台：

1.  **原生兼容**：由于 E4B 原生支持 `<|channel>thought` 这种标签，你可以在微调时，把你的“自定义推理流程”直接塞进这个标签内。
2.  **强制引导**：在微调数据中，你可以通过 system prompt 明确要求模型：“在 `<|channel>thought` 区域内必须包含‘关键词提取’和‘语义转折分析’两个步骤”。
3.  **应用部署**：在推理时，你只需给 E4B 发送 `<|think|>` 指令，它就会乖乖地按照你微调出的“标准动作”进行思考。

---

# 🧠 Gemma 4 自定义思维链（CoT）微调规范

在 Gemma 4 E4B 模型中，实现自定义推理不仅是逻辑上的升级，更是对模型原生 `<|think|>` 控制位的使用优化。通过微调，你可以将模型从“直觉反应”改造为“逻辑推演”模式。

## 🛠️ 1. 数据结构规范化 (SFT Dataset Design)

为了利用 Gemma 4 的原生推理能力，建议在 `completion` 中采用其标准的通道标记格式。这能确保微调后的模型在原生推理接口下表现最稳。



### 规范化数据示例
```python
{
    "prompt": [
        {"role": "system", "content": "你是一个情感分析专家。请按以下流程思考：1.提取关键词 2.分析语义 3.给出最终结论。"},
        {"role": "user", "content": "Classify the emotion: '虽然下雨了，但看到彩虹那一刻我释怀了。'"}
    ],
    "completion": [
        {
            "role": "assistant",
            "content": """<|channel>thought
1. 关键词提取：下雨（负面背景）、彩虹（正面转折）、释怀（核心情感）。
2. 语义分析：句子采用了“虽然...但...”的转折结构，重点在于后半句的情感释放。
3. 结论推导：情感从压抑转向积极。
<channel|>Final Answer: joy"""
        }
    ]
}
```
> **规范要点**：使用 `<|channel>thought` 和 `<channel|>` 作为思考边界，这是 Gemma 4 识别内部推理逻辑的标准占位符。

---

## 🚀 2. 推理阶段的适配规范

当模型学会了上述流程后，推理代码需要精准控制“思维开关”。

### A. 触发与生成控制
利用 `add_generation_prompt` 引导模型进入生成状态，并调优解码参数。

```python
# 推理规范代码
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True, # 必须为 True 以引导 Assistant 开口
    return_tensors="pt"
).to("cuda")

outputs = model.generate(
    inputs,
    max_new_tokens=512,      # 必须预留足够的长度给思维链
    temperature=0.4,         # 推理任务建议降低温度，保持逻辑一致性
    do_sample=True,
    top_p=0.9
)
```

### B. 结构化解析 (Output Parsing)
由于输出包含思维过程，应用端需要通过分隔符进行切分：

```python
raw_output = tokenizer.decode(outputs[0], skip_special_tokens=False)

# 规范化提取逻辑
if "<|channel>thought" in raw_output:
    parts = raw_output.split("<channel|>")
    thinking_process = parts[0].split("<|channel>thought")[-1].strip()
    final_answer = parts[1].strip()
else:
    thinking_process = "直接输出（未触发深度思考）"
    final_answer = raw_output.strip()
```

---

## 💡 3. 核心优势与价值

* **逻辑确定性 (Logic Determinism)**：通过微调 `E4B`，你将原本不可见的“黑盒”计算过程转化为了可观测的“白盒”推理步骤。
* **误差回溯 (Error Backtracing)**：当分类结果为 `sadness` 时，你可以回溯其 `thought` 通道，查看是否是因为漏掉“彩虹”这个关键词导致的错误。
* **原生兼容性**：使用标准标签微调后的模型，可以无缝接入支持 Gemma 4 Thinking 模式的推理框架（如 vLLM 或 llama.cpp）。

---

## ⚠️ 4. 实施进阶建议

1.  **数据质量 > 数量**：对于 E4B 这种中量级模型，**500-1000 条**高质量、逻辑严密的 CoT 数据远比 1 万条纯标签数据有效。
2.  **负面样本强化**：在训练集中加入一些容易误判的样本（如反讽、双重否定），并刻意在 `thought` 中展示模型如何纠正这些误导信息。
3.  **损失函数对齐**：在微调时，确保对 `thought` 部分的 Token 计算 Loss，这样模型才会真正学习“如何思考”，而不仅仅是学习“如何输出标签”。

通过这种方式，你的 Gemma 4 E4B 将不再只是一个分类器，而是一个具备**结构化思维能力**的小型专家系统。

In [ ]:
def to_prompt_completion(example):
    text = example["text"]
    label = label_names[example["label"]]
    return {
        "prompt": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT,
            },
            {
                "role": "user",
                "content": f"Classify the emotion of this text:\n\n{text}",
            },
        ],
        "completion": [
            {
                "role": "assistant",
                "content": label,
            }
        ],
    }

sft_dataset = dataset.map(to_prompt_completion, remove_columns=dataset["train"].column_names)


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

执行 `sft_dataset["train"][0]` 后，你会看到数据已经从原始的扁平结构演变为一个**结构化的嵌套字典**。这是大模型微调中标准的 **Instruction-Response（指令-响应）** 格式。

以下是对输出结构的详细解读：

---

## 🧩 格式化后的样本深度解析

输出的内容大致如下（以第一条数据为例）：

```python
{
    'prompt': [
        {'role': 'system', 'content': 'You are an emotion classification assistant...'},
        {'role': 'user', 'content': 'Classify the emotion of this text:\n\ni didnt feel humiliated'}
    ],
    'completion': [
        {'role': 'assistant', 'content': 'sadness'}
    ]
}
```

### 1. 结构化拆解
* **`prompt` 列表**：包含了模型在推理时需要看到的“上下文”。
    * **System Role**：确立了 Gemma 4 的专家身份和输出规则。
    * **User Role**：提供了具体的待处理文本和任务指令。
* **`completion` 列表**：包含了模型应该生成的“标准答案”。
    * **Assistant Role**：这是训练时的**正样本标签**。模型的目标就是学习在看到上面的 `prompt` 后，精准地预测出 `sadness` 这个词。

### 2. 为什么这是微调的“黄金格式”？
这种格式完美适配了 **TRL (Transformer Reinforcement Learning)** 库中的 `SFTTrainer`。
* **Prompt 与 Completion 分离**：这种设计允许训练器在计算损失（Loss）时，**仅计算 Completion 部分的梯度**。
* **逻辑清晰**：模型不会把你的指令（Classify the emotion...）当成它需要学习生成的文本，它只会学习如何给出答案。

---

## 💡 微调 Gemma 4 的核心逻辑：从文本到 Token

虽然你现在看到的是人类可读的文字，但在喂给 Gemma 4 之前，后台还会发生一次隐形的转换。



### 3. 下一步：Chat Template 的自动填充
当你后续调用 `tokenizer.apply_chat_template` 时，这个字典会被自动包装成 Gemma 4 专属的特殊标记格式。例如：

```text
<start_of_turn>user
You are an emotion classification assistant...
Classify the emotion of this text:
i didnt feel humiliated<end_of_turn>
<start_of_turn>model
sadness<end_of_turn>
```

---

### 🔎 检查要点
当你运行这一行并检查输出时，请务必确认：
1.  **标签是否正确**：确认 `completion` 里的单词是否真的对应原始数据的情感（比如索引 0 是否正确映射为了 `sadness`）。
2.  **换行符是否生效**：确认用户输入中的 `\n\n` 是否按预期分开了指令和样本。
3.  **无冗余列**：确认输出中**没有**出现旧的 `text` 或 `label` 字段（这说明 `remove_columns` 参数执行成功了）。

In [ ]:
sft_dataset["train"][0]


{'prompt': [{'role': 'system',
   'content': "You are an emotion classification assistant.\nRead the user's text and answer with exactly one label.\nOnly choose from: sadness, joy, love, anger, fear, surprise.\nReturn only the label and nothing else."},
  {'role': 'user',
   'content': 'Classify the emotion of this text:\n\nwhile cycling in the country'}],
 'completion': [{'role': 'assistant', 'content': 'fear'}]}

## 6. 加载处理器和基础模型，用于微调前评估

这段代码是微调流程中**技术含量最高**的环节之一：**模型加载与显存优化配置**。它决定了你是否能在有限的硬件资源（如 Colab 的 T4 或 L4 GPU）上跑动庞大的 Gemma 4 模型。

---

## 🏗️ 模型加载与 4-bit 量化架构解析

### 1. 硬件加速策略 (TF32)
```python
if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
```
* **TF32 (TensorFloat-32)**：这是针对 NVIDIA Ampere 架构及以后（如 A100, L4）的优化。它在保持接近 FP32 精度的同时，提供类似 FP16 的计算速度。开启它能显著提升矩阵乘法的效率。

### 2. 分词器 (Tokenizer/Processor) 配置
* **`use_fast=True`**：启用基于 Rust 实现的快速分词器，处理大规模数据集时速度极快。
* **`pad_token` 设置**：由于 Gemma 等生成式模型默认可能没有“填充符（Pad Token）”，这里将其指向“结束符（EOS Token）”。这是为了在批处理训练时，让不等长的句子能对齐，同时不引入新的干扰 Token。

### 3. 核心魔法：BitsAndBytes 4-bit 量化
这是在消费级显卡上运行 Gemma 4 的关键。
* **`load_in_4bit=True`**：将模型权重从 16 位压缩到 4 位。这能将模型占用的显存减少约 **70%**。
* **`bnb_4bit_quant_type="nf4"`**：使用 **NormalFloat 4** 数据类型。这是专门为正态分布的模型权重设计的，比普通的 4-bit 量化精度更高。
* **`bnb_4bit_compute_dtype=torch.bfloat16`**：虽然权重存成 4-bit，但计算时会临时转回 **bfloat16**。这能避免精度大幅下降，并利用现代 GPU 对 BF16 的原生支持。



### 4. 自动化设备映射 (`device_map="auto"`)
* 这一行代码会自动检测你的硬件。如果你有多块 GPU，它会自动做模型切分；如果显存不足，它甚至会将部分层卸载到 CPU。这消除了手动配置 `device` 的麻烦。

### 5. 状态与配置同步
```python
base_model.config.use_cache = False
```
* **`use_cache = False`**：**微调时必须关闭**。Cache 是为了加速推理生成（预测下一个词时缓存之前的状态），但在训练（计算梯度）过程中，开启它会引发冲突且消耗额外显存。
* **Token ID 同步**：手动将模型配置和生成配置中的 `pad`, `bos`, `eos` ID 与分词器对齐，确保模型知道什么时候开始、什么时候闭嘴。

---

## 💡 为什么这对 Gemma 4 特别重要？

1.  **显存预算**：Gemma 4 作为一个高性能模型，如果以全精度（FP32）加载，可能需要几十 GB 显存。使用 4-bit 量化后，原本需要 20GB+ 的模型，现在大约 5-8GB 就能跑起来，这让在 Colab 免费版上微调成为可能。
2.  **训练稳定性**：使用 `bfloat16` (BF16) 代替标准的 `float16` 可以有效防止训练中的梯度溢出（Overflow）问题，这在微调深层大模型时非常关键。

---

### 🔎 检查输出
当看到 `Base model loaded with 4-bit=True and dtype=torch.bfloat16.` 时，说明你的“引擎”已经成功安装在底盘上，并且已经通过压缩技术塞进了有限的机舱里。

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "google/gemma-4-E4B-it"
MODEL_DTYPE = torch.bfloat16
USE_4BIT = True

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

processor = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if processor.pad_token is None:
    processor.pad_token = processor.eos_token

bnb_config = None
model_kwargs = {
    "device_map": "auto",
}
if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=MODEL_DTYPE,
    )
    model_kwargs["quantization_config"] = bnb_config
else:
    model_kwargs["torch_dtype"] = MODEL_DTYPE

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
base_model.config.use_cache = False
base_model.config.pad_token_id = processor.pad_token_id
base_model.config.bos_token_id = processor.bos_token_id
base_model.config.eos_token_id = processor.eos_token_id
base_model.generation_config.pad_token_id = processor.pad_token_id
base_model.generation_config.bos_token_id = processor.bos_token_id
base_model.generation_config.eos_token_id = processor.eos_token_id

print(f"Base model loaded with 4-bit={USE_4BIT} and dtype={MODEL_DTYPE}.")


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Base model loaded with 4-bit=True and dtype=torch.bfloat16.


## 7. 用于 Gemma 4 对话格式的推理辅助函数

这段代码是微调流程中的**评估与推理引擎**。它的作用是让模型真正“跑”起来，并利用正则表达式（Regex）对模型生成的原始文本进行“硬约束”清洗，确保最终输出的是标准化的标签。

---

## 🛠️ 推理与后处理逻辑解析

### 1. 鲁棒的标签提取 (`extract_label`)
即使我们在 System Prompt 中要求模型“仅返回标签”，大模型有时仍会产生多余的空格、标点或语气词（如 "The emotion is joy."）。
* **`LABEL_PATTERN`**: 使用正则表达式 `\b...\b` 进行边界匹配。这意味着它能从 `"The label is joy!"` 中精准抓取 `joy`，而不会被其他字符干扰。
* **`re.IGNORECASE`**: 忽略大小写，增强了对模型不同输出习惯的兼容性。
* **后备逻辑**: 如果正则匹配失败，代码会取输出的第一个单词并剔除标点符号。这是一种双重保险机制。

### 2. 对话模板应用 (`apply_chat_template`)
这是处理 Gemma 4 等指令模型最标准、最推荐的方式：
* **`add_generation_prompt=True`**: 这会自动在 Prompt 末尾添加模型特有的“助手开始说话”标记（如 `<start_of_turn>model\n`）。如果没有这一行，模型可能不知道轮到它发言了。
* **`return_tensors="pt"`**: 直接返回 PyTorch 张量，准备好输入显卡。

### 3. 精准生成控制 (`model.generate`)
* **`max_new_tokens=4`**: **非常聪明的优化！** 因为我们只需要模型输出一个情感词（如 `surprise`），设置极小的 Token 限制可以显著加快推理速度，并防止模型开启“废话模式”。
* **`do_sample=False`**: 开启**贪婪搜索（Greedy Search）**。在分类任务中，我们希望结果是确定的、最高概率的，而不是具有随机性的创作。
* **`input_len` 截断**: 在解码时使用 `outputs[0][input_len:]`，这确保了我们只解码模型**新生成**的内容，而不是把用户输入的 Prompt 也重新打印一遍。



---

## 💡 为什么在微调 Gemma 4 时需要这套逻辑？

### A. 弥补“指令遵循”的偶发失效
即便模型经过了微调，在面对某些复杂文本时，它仍可能输出 `Label: joy`。通过 `extract_label`，我们可以将这种不规范的输出强行拉回 `joy`，确保评估指标（如准确率）计算的公正性。

### B. 显存与速度平衡
通过 `torch.no_grad()` 装饰器，我们告诉 PyTorch 不需要计算梯度，这能节省大量显存并加速推理。配合 `max_new_tokens=4`，你可以快速地在整个测试集（如 400 条数据）上完成预测。

### C. 跨设备兼容性
代码中的 `device = next(model.parameters()).device` 确保了无论你的模型是在 CPU、单卡 GPU 还是多卡分布式环境下，推理代码都能自动找到正确的计算位置。

---

## 🔍 代码运行流程速览
1.  **输入**: "i feel amazing today"
2.  **Prompt 转换**: 变成带有特殊标记的 Gemma 格式。
3.  **模型预测**: 模型输出原始 Token IDs。
4.  **解码**: 得到字符串，例如 `"  Joy."`。
5.  **清洗**: `extract_label` 介入，输出最终结果 `"joy"`。

In [ ]:
import re

LABEL_PATTERN = re.compile(r"\b(sadness|joy|love|anger|fear|surprise)\b", re.IGNORECASE)

def extract_label(raw_text: str) -> str:
    raw_text = raw_text.strip().lower()
    match = LABEL_PATTERN.search(raw_text)
    if match:
        return match.group(1)

    first_token = raw_text.split()[0].strip(".,!?:;\"'()[]{}") if raw_text.split() else ""
    return first_token

def generate_label(model, processor, user_text, system_prompt, max_new_tokens=4):
    messages = [
        {
            "role": "system",
            "content": system_prompt,
        },
        {
            "role": "user",
            "content": f"Classify the emotion of this text:\n\n{user_text}",
        },
    ]

    device = next(model.parameters()).device
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(device)

    input_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=processor.pad_token_id,
            eos_token_id=processor.eos_token_id,
        )

    raw_pred = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    return extract_label(raw_pred)


这段代码定义了一个高层级的**推理入口函数**，它将复杂的模型调用逻辑封装成了一个简单的单行接口，用于快速测试模型的分类效果。

---

## 🛠️ 推理封装函数 `predict_emotion` 解析

### 1. 默认对象处理逻辑
* **`model = model or base_model`** 与 **`proc = proc or processor`**：
    这是 Python 中常用的**延迟绑定**策略。它允许函数在不传入特定参数时，自动使用全局作用域中已经加载好的 `base_model` 和 `processor`。
    * **灵活性**：如果你后续加载了微调后的模型（例如 `peft_model`），你可以通过传参来对比不同模型的效果；如果不传参，它默认指向你当前内存中的基础模型。

### 2. 核心逻辑解耦
* **`return generate_label(...)`**：
    该函数本身不处理底层的 Tokenization 或设备映射，而是作为一个**代理（Proxy）**调用你之前定义的 `generate_label`。这种解耦设计使得你可以全局统一修改推理行为（如修改 `max_new_tokens`），而无需逐个修改业务调用点。

### 3. 自动注入 System Prompt
* 函数内部直接引用了全局变量 `SYSTEM_PROMPT`。这意味着当你调用 `predict_emotion("文本")` 时，函数会自动在后台为这段文本穿上“外壳”，即：
    > "You are an emotion classification assistant... [用户文本]"
    
    这确保了用户在测试时只需要关注原始文本，而模型接收到的输入始终符合预期的指令格式。

---

## 🔍 代码执行流程演示

当你运行 `predict_emotion("I feel so happy and excited today!")` 时，后台发生了以下链式反应：

1.  **构造对话流**：将输入文本与系统提示词组合成标准对话结构。
2.  **模板转换**：调用 `processor.apply_chat_template` 插入 Gemma 4 的特殊标记。
3.  **模型推理**：`base_model` 在 `torch.no_grad()` 模式下进行 4-bit 量化加速运算，生成最多 4 个新 Token。
4.  **文本解码**：将输出 ID 转回字符串（如 `"  joy"`）。
5.  **正则清洗**：`extract_label` 提取出最终的纯净标签。
6.  **结果返回**：返回字符串 `"joy"`。



---

## 💡 技术亮点

* **极简接口**：将涉及显存管理、设备映射、正则匹配和模板填充的复杂流程，简化为 `f(text) -> label` 的形式，非常适合在数据科学笔记本（Notebook）中进行交互式验证。
* **语义一致性**：通过强制注入 `SYSTEM_PROMPT`，保证了推理时的环境与训练时的环境逻辑高度闭环，避免了“训练与推理不一致”导致的性能下降。

在 Colab 中运行这段代码时，输出结果包含了一段**警告信息**和最终的**预测结果**。作为模型微调前的性能基准测试，这种输出反映了预训练模型在原生状态下的行为。

以下是针对这两个部分的详细解释：

### 1. 警告信息：Generation Flags Ignoring(微调前)
> `[transformers] The following generation flags are not valid and may be ignored: ['top_p', 'top_k'].`

这段警告来自于 Hugging Face 的 `transformers` 库，其核心原因在于**配置冲突**：

* **冲突来源**：在之前的 `generate_label` 函数中，你显式设置了 `do_sample=False`。这告诉模型使用**贪婪搜索（Greedy Search）**，即每次只选择概率最高的 Token。
* **参数失效**：`top_p`（核采样）和 `top_k`（顶级采样）是专门用于**随机采样（Sampling）**模式的参数。当你禁用了采样（`do_sample=False`）时，这些参数在逻辑上就不再起作用。
* **为什么会出现**：Gemma 4 的默认配置（Generation Config）中通常自带了这些采样参数。当你调用 `model.generate` 时，库会发现你既关闭了采样又保留了采样参数，因此自动忽略它们并发出提醒。这**不会影响**预测结果的准确性。

### 2. 预测结果：`joy`(微调后)
这是 `predict_emotion` 函数返回的最终字符串，代表了模型对输入文本的情感分类。

* **零样本能力（Zero-shot Capability）**：
    由于此时尚未开始微调，这个 `joy` 反映了 Gemma 4 强大的**原生语义理解能力**。它通过预训练阶段习得的知识，准确识别出 "happy" 和 "excited" 属于 "joy"（喜悦）这一维度。
    
* **指令遵循（Instruction Following）**：
    输出直接是 `joy` 而不是长篇大论，说明你在函数中定义的 `SYSTEM_PROMPT` 起到了很好的约束作用。模型理解了“仅返回一个情感标签”的指令。

* **格式清理（Regex Post-processing）**：
    输出结果显得非常干净，归功于你在 `generate_label` 中嵌套调用的 `extract_label` 函数。即便是模型在生成时带了一些多余的空格或换行符，正则表达式也将其精准地修剪成了纯净的标签字符串。


In [ ]:
def predict_emotion(user_text: str, model=None, proc=None) -> str:
    model = model or base_model
    proc = proc or processor
    return generate_label(model, proc, user_text, SYSTEM_PROMPT)

predict_emotion("I feel so happy and excited today!")

[transformers] The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'joy'

## 8. 评估辅助函数

这段代码构建了一个完整的**模型性能评估框架**。它不仅计算标准的机器学习指标，还专门针对大语言模型（LLM）可能产生的非标准输出设计了容错机制。

---

## 📊 评估框架与指标计算解析

### 1. 容错与异常处理机制
* **`VALID_LABELS`**: 将标签列表转换为集合（Set），用于提升成员检查的效率。
* **`INVALID` 状态**: 这是针对生成式模型（Generative AI）的特殊处理。如果模型生成的词不在预定义的 6 个情感类别中，它会被标记为 `"INVALID"` 而不是让代码报错崩溃。
* **`ALL_EVAL_LABELS`**: 在混淆矩阵中加入 `"INVALID"` 列，可以直观地看到模型在哪些样本上由于“胡言乱语”而失去了分类能力。

### 2. 评估核心逻辑 (`evaluate_model`)
* **`model.eval()`**: 将模型切换至**评估模式**。这会关闭 Dropout 和 Batch Normalization 等在训练时才需要的随机行为，确保预测结果的确定性和稳定性。
* **`tqdm` 进度条**: 在处理数以百计的测试样本时，提供实时的可视化进度反馈，展示预测速度和剩余时间。
* **数据收集 (`rows`)**: 代码不仅记录结果，还通过字典 `rows` 保存了原始文本、预测值和真实值的对应关系。这为后续分析模型“为什么会预测错”提供了原始素材。

### 3. 指标衡量维度
* **`accuracy_score`**: 计算整体准确率，即预测对的样本占总样本的比例。
* **`macro_f1` (宏 F1 分数)**:
    * **`average="macro"`**: 对所有类别一视同仁地计算 F1 并取平均。由于情感分布可能不均，宏平均能比准确率更客观地反映模型对小众类别（如 `surprise`）的捕捉能力。
    * **`zero_division=0`**: 这是一个健壮性设置。如果模型从未预测过某个类别，该参数防止程序抛出除以零的数学错误。
* **`invalid_predictions`**: 统计模型不按套路出牌的频率，是衡量微调后指令遵循能力的重要指标。



### 4. 数据透视与混淆矩阵 (`confusion_matrix_df`)
* **`confusion_matrix`**: 利用 `sklearn` 生成原始矩阵，展示真实标签与预测标签的交叉分布。
* **`pd.DataFrame` 包装**: 将纯数字矩阵转换为带标签的 DataFrame，横轴为预测值，纵轴为真实值。
    * **对角线**：代表预测正确的样本。
    * **非对角线**：代表模型容易混淆的类别（例如：模型是否经常把 `love` 误判为 `joy`）。

---

## 💡 技术亮点

* **端到端闭环**：该函数完成了从模型输入、文本生成、正则解析、到最终统计报表生成的全过程。
* **精细化分析**：通过返回 `df` (DataFrame)，用户可以轻松筛选出所有 `correct == False` 的行，进行**错误分析 (Error Analysis)**，这是优化微调策略（如调整 Prompt 或增加特定数据）的关键依据。
* **针对 LLM 优化**：区别于传统的分类器，它深刻考虑到了生成式模型输出的不确定性，通过 `INVALID` 标签和 `labels` 约束确保了评估的科学性。

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
import pandas as pd
from tqdm.auto import tqdm

VALID_LABELS = set(label_names)
ALL_EVAL_LABELS = label_names + ["INVALID"]

def evaluate_model(model, processor, split="test", limit=EVAL_LIMIT):
    y_true, y_pred, rows = [], [], []
    raw_source = dataset[split]
    if limit is not None:
        raw_source = raw_source.select(range(min(limit, len(raw_source))))

    model.eval()

    for ex in tqdm(raw_source, desc=f"Evaluating {split}", leave=False):
        true_label = label_names[ex["label"]]
        raw_pred_label = generate_label(model, processor, ex["text"], SYSTEM_PROMPT)
        pred_label = raw_pred_label if raw_pred_label in VALID_LABELS else "INVALID"

        y_true.append(true_label)
        y_pred.append(pred_label)
        rows.append({
            "text": ex["text"],
            "true_label": true_label,
            "pred_label": pred_label,
            "raw_pred_label": raw_pred_label,
            "correct": true_label == pred_label,
        })

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=label_names, average="macro", zero_division=0),
        "invalid_predictions": sum(1 for p in y_pred if p == "INVALID"),
        "evaluated_examples": len(y_true),
    }

    report = classification_report(
        y_true,
        y_pred,
        labels=label_names,
        output_dict=True,
        zero_division=0,
    )

    df = pd.DataFrame(rows)
    return metrics, report, df

def confusion_matrix_df(pred_df):
    return pd.DataFrame(
        confusion_matrix(pred_df["true_label"], pred_df["pred_label"], labels=ALL_EVAL_LABELS),
        index=ALL_EVAL_LABELS,
        columns=ALL_EVAL_LABELS,
    )


## 9. 微调前评估

这两行代码启动了**微调前的基准评估（Baseline Evaluation）**。通过在测试集上运行评估框架，你可以量化 Gemma 4 在未接触特定领域数据时的“原生”情感分析能力。

---

## 📈 基准评估执行与指标解析

### 1. 评估执行过程 (`evaluate_model`)
当这一行运行时，程序会执行以下操作：
* **数据切片**：从 `dataset["test"]` 中提取样本（由之前的 `EVAL_LIMIT` 限制，通常为 400 条）。
* **批量预测**：对于每一条测试文本，模型都会生成一次预测。由于使用的是 `base_model`（加载了 4-bit 量化且尚未训练），这反映了 Google 原厂模型的 **Zero-shot（零样本）** 性能。
* **数据对齐**：将预测结果与真实标签进行比对。

### 2. 返回值对象说明
* **`pre_metrics`**：包含整体性能的摘要字典（准确率、F1 分数等）。
* **`pre_report`**：包含每个具体类别（如 `joy` vs `anger`）的精确率、召回率的详细字典。
* **`pre_preds`**：一个 Pandas DataFrame，记录了每一条测试数据的详细结果，方便你后续通过 `pre_preds.head()` 查看具体哪些句子预测错了。

### 3. `pre_metrics` 预期输出字段
运行 `pre_metrics` 后，你会看到一个类似以下的字典：
* **`accuracy`**：模型猜对的比例。
* **`macro_f1`**：综合考虑了各类别的平衡得分。通常预训练模型在没有微调的情况下，这个分数会显著低于微调后的模型。
* **`invalid_predictions`**：这个数值非常关键。它反映了原厂模型在**不微调**的情况下，遵守“仅输出标签”指令的能力。如果这个值很高，说明模型虽然聪明但“不听话”。
* **`evaluated_examples`**：确认实际评估的样本数量。

---

## 💡 为什么微调前必须运行这一步？

* **科学对比**：微调的目标通常是提升准确率或降低 `invalid_predictions`。如果没有这个 `pre_metrics` 作为对照组，你将无法量化微调到底带来了多少提升。
* **验证任务难度**：如果 `base_model` 的初始准确率就已经非常高（例如 >85%），那么可能不需要大规模微调；如果准确率很低，则说明该特定任务的标签定义与模型的预训练常识存在偏差。
* **调试后处理**：通过查看 `pre_metrics` 中的 `invalid_predictions`，你可以判断之前的正则表达式 `extract_label` 是否足够强大，能够处理模型可能给出的各种奇怪回复。

In [ ]:
pre_metrics, pre_report, pre_preds = evaluate_model(base_model, processor, "test")
pre_metrics


Evaluating test:   0%|          | 0/400 [00:00<?, ?it/s]

{'accuracy': 0.575,
 'macro_f1': 0.4170383807826307,
 'invalid_predictions': 38,
 'evaluated_examples': 400}

这段代码利用 **Pandas** 将复杂的评估报告字典转换为一个**结构化的表格**。这是数据科学中进行“错误分析”和“模型诊断”的标准操作。

---

## 📊 评估报告表格化解析

### 1. `pd.DataFrame(pre_report)`
* **操作内容**：将 `classification_report` 生成的嵌套字典转换为 DataFrame。
* **默认形态**：默认情况下，Pandas 会将字典的第一层键（即 `sadness`, `joy` 等类别名称）作为**列名（Columns）**，而将指标（`precision`, `recall`, `f1-score`, `support`）作为**行索引（Rows）**。
* **局限性**：这种“宽表”格式在类别较多时很难阅读，不符合人类观察分类报表的习惯。

### 2. `.transpose()` (或 `.T`)
* **操作内容**：执行**矩阵转置**，将行与列互换。
* **结果形态**：转置后，每一行代表一个具体的情感类别，每一列则代表一个核心评估维度。

---

## 🔍 表格字段深度解析

转置后的表格通常包含以下列，它们从不同维度刻画了 Gemma 4 的原生能力：

| 指标列名 | 含义解析 | 在 Gemma 4 微调背景下的意义 |
| :--- | :--- | :--- |
| **`precision`** | **精确率**：预测为该情感的样本中，实际也为该情感的比例。 | 衡量模型是否“乱猜”。如果 `love` 的精确率低，说明模型容易把别的情感错判为爱。 |
| **`recall`** | **召回率**：实际为该情感的样本中，被模型正确预测出来的比例。 | 衡量模型是否“漏判”。如果 `fear` 的召回率低，说明模型对恐惧情感不够敏感。 |
| **`f1-score`** | **F1 分数**：精确率与召回率的调和平均数。 | **最核心的综合指标**。它能防止模型通过只预测某个高频类别（如 `joy`）来刷高分。 |
| **`support`** | **样本数**：该类别在测试集中的实际数量。 | 让你意识到数据的分布情况。如果某个类别只有 10 个样本，其高 F1 分数的统计意义较弱。 |



### 3. 特殊行解析
在表格的底部，通常会出现三行全局统计数据：
* **`accuracy`**：全量样本的准确率（通常只有一行，其他列为空）。
* **`macro avg` (宏平均)**：对所有类别的指标取简单算术平均。它对所有类别一视同仁，最能体现模型在**不平衡数据集**上的真实功力。
* **`weighted avg` (加权平均)**：根据每个类别的样本数（support）进行加权平均。它更倾向于反映模型在常见情感上的表现。

---

## 💡 为什么这个表格对微调至关重要？

通过这个表格，你可以精准定位基准模型的**“弱点”**。

例如，如果你发现 `surprise` 类别的 F1 分数为 0，或者 `precision` 极低，这说明 Gemma 4 的原始知识可能无法很好地界定什么是“惊讶”。在接下来的 **PEFT/LoRA 微调**中，你就需要重点观察这些弱势类别的指标是否得到了提升，从而验证微调的有效性。

In [ ]:
pd.DataFrame(pre_report).transpose()

,precision,recall,f1-score,support
sadness,0.562130,0.833333,0.671378,114.0
joy,0.792857,0.685185,0.735099,162.0
love,0.285714,0.230769,0.255319,26.0
anger,0.750000,0.063830,0.117647,47.0
fear,0.647059,0.305556,0.415094,36.0
surprise,0.363636,0.266667,0.307692,15.0
micro avg,0.635359,0.575000,0.603675,400.0
macro avg,0.566899,0.397557,0.417038,400.0
weighted avg,0.659882,0.575000,0.568374,400.0


这段代码通过调用之前定义的 `confusion_matrix_df` 函数，生成并展示了基准模型的**混淆矩阵（Confusion Matrix）**。这是评估分类模型表现最直观、最细致的工具。

---

## 🏁 混淆矩阵表格化解析

### 1. 矩阵结构说明
该矩阵是一个 **(N+1) x (N+1)** 的方阵（N 为情感类别数，本例中为 6 个类别 + 1 个 `INVALID` 类别）：
* **行（Rows/Index）**：代表数据的**真实标签（True Labels）**。
* **列（Columns）**：代表模型的**预测标签（Predicted Labels）**。
* **单元格 (i, j)**：表示有多少个属于类别 $i$ 的样本被模型预测为了类别 $j$。

### 2. 关键区域解读
* **主对角线（Diagonal）**：从左上角到右下角的数值。这些是预测正确的样本。数值越高，说明模型对该类别的识别越准确。
* **非对角线区域（Off-diagonal）**：代表预测错误的情况。
    * 如果某一行除了对角线外，在另一列有很高的数值，说明模型**经常混淆**这两个类别（例如：经常把 `anger` 误判为 `fear`）。
* **`INVALID` 列**：这是一个针对 LLM 的特殊观察哨。如果该列数值较高，说明模型即便有系统提示词的约束，依然无法稳定输出预定义的标签，存在严重的“不听话”现象。



---

## 💡 技术价值

### A. 诊断“语义重叠”
在情感分析中，某些类别在语义上非常接近（如 `joy` 和 `love`）。混淆矩阵能帮你确认 Gemma 4 在原生状态下是否具备区分这些细微差别的能力。如果矩阵显示大量 `love` 被判为了 `joy`，在后续微调时，你可能需要关注是否需要更多 `love` 类别的训练样本。

### B. 指导微调策略
通过观察 `INVALID` 行和列：
* **如果是全 0**：说明 `base_model` 的指令遵循能力极强。
* **如果有数值**：说明模型在特定语境下会失效。微调的一个核心目标就是通过 **SFT（有监督微调）** 将这些分散在非对角线和 `INVALID` 列的数值“驱赶”到主对角线上。

### C. 确定基准线
这是你微调前的最后一张“体检表”。在完成 LoRA 训练后，你会再次生成这张表。两张表对比，能最清晰地展现微调是如何修复模型的认知偏差并强化其分类边界的。

In [ ]:
confusion_matrix_df(pre_preds)


,sadness,joy,love,anger,fear,surprise,INVALID
sadness,95,8,3,1,2,1,4
joy,25,111,7,0,2,4,13
love,8,11,6,0,0,0,1
anger,22,3,2,3,2,0,15
fear,14,4,1,0,11,2,4
surprise,5,3,2,0,0,4,1
INVALID,0,0,0,0,0,0,0


## 10. 配置 LoRA 适配器

这段代码标志着微调过程进入了最核心的**配置阶段**。它定义了 **LoRA (Low-Rank Adaptation)** 的参数，这是实现“轻量化微调”的关键，允许你在不改动模型几十亿个原始参数的情况下，通过训练微小的“旁路”矩阵来让模型学会新任务。

---

## 🏗️ LoRA 配置参数深度解析

### 1. `r=16` (秩, Rank)
* **技术含义**：这是低秩矩阵的维度。LoRA 的核心思想是将参数更新量 $\Delta W$ 分解为两个极小的矩阵 $A$ 和 $B$ 的乘积（即 $\Delta W = A \times B$）。
* **影响**：`r` 值决定了可训练参数量的多少。`16` 是一个兼顾了模型表现和计算效率的平衡值。秩越高，模型捕捉复杂任务特征的能力越强，但显存消耗也会随之增加。

### 2. `lora_alpha=32` (缩放系数)
* **技术含义**：这是一个缩放因子，用于调整 LoRA 权重对原始模型影响的权重。
* **计算逻辑**：实际应用的增量权重会乘以 $\frac{\alpha}{r}$。在本例中，缩放比为 $32/16 = 2$。
* **作用**：调大此值可以让模型更积极地学习新任务的数据分布，但过大可能导致模型遗忘原有的通用能力。

### 3. `lora_dropout=0.05`
* **技术含义**：在训练过程中，随机将 5% 的 LoRA 神经元失活。
* **作用**：这是一种正则化手段，目的是防止模型在 4000 条训练数据上产生**过拟合**。它能强迫模型学习更具泛化性的特征，而不是简单地死记硬背训练集的标签。

### 4. `target_modules="all-linear"`
* **技术含义**：指定要在模型的哪些层中插入 LoRA 旁路。
* **优势**：
    * **全面性**：Gemma 4 拥有复杂的注意力机制和前馈网络。指定 `"all-linear"` 意味着会在所有的线性层（如 `q_proj`, `k_proj`, `v_proj`, `o_proj` 以及 `gate_proj`, `up_proj`, `down_proj`）中应用 LoRA。
    * **性能提升**：研究表明，在模型的所有线性层上应用 LoRA，通常比只在 Attention 层应用能获得更好的微调效果。



### 5. `task_type="CAUSAL_LM"`
* **技术含义**：明确告诉 PEFT 库，这是一个**因果语言模型**（即根据上文预测下文的生成式模型）。
* **作用**：这确保了 LoRA 能够正确挂载到 Gemma 4 这种从左到右生成的架构上。

---

## 💡 技术亮点

* **极高的显存效率**：通过这种配置，你需要训练的参数量通常不到原始模型参数总量的 **1%**。这意味着原本需要昂贵 A100 显卡才能完成的任务，现在可以在普通的 T4/L4 显卡上轻松运行。
* **模块化更新**：微调结束后，你只需要保存这些只有几十 MB 的 LoRA 权重文件，而不需要重新保存几十 GB 的完整模型。
* **适配 Gemma 4**：针对 Gemma 4 这种层数深、参数分布广的模型，使用 `"all-linear"` 是一种非常稳健的选择，能够确保模型在理解情感语义时能够覆盖到网络的所有关键转换环节。

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear"
)


## 11. 定义训练配置

这段代码使用了 **TRL (Transformer Reinforcement Learning)** 库中的 `SFTConfig` 来配置**有监督微调（Supervised Fine-Tuning）**的所有超参数。这是控制训练过程、平衡计算速度与模型质量的关键配置文件。

---

## ⚙️ 训练超参数与配置深度解析

### 1. 显存管理与计算效率
* **`gradient_accumulation_steps=2`**：梯度累积。它允许你在有限的显存下模拟更大的 Batch Size。实际的逻辑 Batch Size 是 `batch_size (8) * accumulation_steps (2) = 16`。这有助于训练过程更加平稳。
* **`gradient_checkpointing=True`**：**显存优化的杀手锏**。它在反向传播时不保存所有的中间激活值，而是在需要时重新计算。虽然会轻微增加计算时间，但能极大地降低显存占用，是微调 Gemma 4 这种大模型的必备设置。
* **`bf16=True`**：启用 Brain Floating Point 16 位精度。相比传统的 `fp16`，`bf16` 动态范围更广，能有效防止梯度爆炸，是 A100/L4 等现代 GPU 训练的最佳选择。

### 2. 优化器与学习率策略
* **`optim="paged_adamw_8bit"`**：使用了 **8-bit 量化优化器**。它将优化器的状态（通常占用大量显存）压缩，并支持“分页”功能（当显存不足时自动溢出到 CPU 内存），确保训练不会因为 OOM（显存溢出）而中断。
* **`learning_rate=1e-4`**：LoRA 微调的经典学习率。比全量微调稍大，因为我们只更新极少数参数。
* **`warmup_steps=50`**：预热步数。在训练初期让学习率从 0 缓慢爬升，防止模型在刚接触数据时因为梯度过大而导致权重被破坏。

### 3. SFT 专用训练设置
* **`completion_only_loss=True`**：**这是微调中最关键的逻辑之一**。它告诉训练器**只计算 Assistant 回答部分的损失**。模型不会因为学习“如何背诵 System Prompt 或用户指令”而浪费梯度空间，而是专注于学习如何给出正确的标签（如 `joy`）。
* **`packing=False`**：不进行序列打包。由于我们的情感分类文本通常很短，关闭打包可以保持每个样本的独立性，避免多条数据混在一起干扰分类逻辑。
* **`max_length=256`**：限制最大序列长度。对于情感分析任务，256 已经绰绰有余，这能进一步优化内存分配。



### 4. 评估与记录
* **`eval_strategy="steps"`**：每隔一定步数（由 `logging_steps` 或默认设置决定）运行一次验证集评估。
* **`metric_for_best_model="eval_loss"`**：以验证集损失作为衡量模型优劣的标准。
* **`remove_unused_columns=False`**：保留所有数据列。这在使用 `SFTTrainer` 处理自定义格式（如 `prompt` 和 `completion`）时非常重要，防止训练器误删必要的文本字段。

---

## 💡 技术要点总结

该配置采用了 **"高效微调三部曲"**：
1.  **量化优化器** (`paged_adamw_8bit`) 降低状态显存。
2.  **重算机制** (`gradient_checkpointing`) 降低激活值显存。
3.  **精准损失计算** (`completion_only_loss`) 提高任务收敛效率。

这套配置确保了即便在 Colab 的普通 GPU 环境下，也能稳定、高效地完成对 Gemma 4 的专业化情感分析改造。

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="./gemma4-emotion-lora",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=50,
    num_train_epochs=1,
    logging_steps=50,
    eval_strategy="steps",
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    bf16=True,
    fp16=False,
    tf32=False,
    max_length=256,
    packing=False,
    completion_only_loss=True,
    remove_unused_columns=False,
    dataloader_num_workers=2,
    optim="paged_adamw_8bit",
    report_to="none",
)


## 12. 训练模型

这段代码是整个微调工程的**终极点火阶段**。它完成了从“配置”到“实战”的转化，将 LoRA 适配器正式挂载到 Gemma 4 模型上，并启动了反向传播训练过程。

---

## 🚀 训练启动与状态管控解析

### 1. 模型状态清理
* **`isinstance(base_model, PeftModel)`**: 这是一个防御性检查。如果你在同一个 Notebook 单元格中多次运行这段代码，它会确保先“卸载”旧的 LoRA 层，回归到一个干净的 `base_model`，防止适配器嵌套导致的显存崩溃或逻辑错误。
* **`unload()`**: 将模型恢复为原始状态，清除之前的微调权重。

### 2. `SFTTrainer` 的核心调度
这是微调的大脑，协调各个组件协同工作：
* **`model=base_model` & `peft_config=lora_config`**: Trainer 会自动根据你定义的 `lora_config` 在模型中注入低秩矩阵。
* **`processing_class=processor`**: 这里的 `processor`（即 Tokenizer）会被用于自动处理 `sft_dataset`。由于你之前已经将数据格式化为 `prompt` 和 `completion` 结构，Trainer 会根据 `training_args` 中的 `completion_only_loss` 自动计算 Mask，确保模型只学习回答部分的 Token。

### 3. 参数完整性验证 (Trainable Params Check)
* **`param.requires_grad`**: 这段循环逻辑是微调前的“安全检查”。
* **意义**：在 LoRA 微调中，原始模型的几十亿个参数都是 `False`（冻结状态），只有新加入的旁路矩阵是 `True`。如果计算出的 `trainable_params` 为 0，说明 `target_modules` 匹配失败（可能是层名称写错了），此时抛出 `RuntimeError` 可以防止你浪费时间训练一个完全不更新的模型。

### 4. 训练执行与收尾
* **`trainer.train()`**: 启动正式训练。此时 GPU 开始全速运转，进行前向传播、损失计算、反向传播和梯度更新。
* **状态切换**：
    * **`trainer.model.eval()`**: 训练结束后，立即切换到评估模式，关闭 Dropout。
    * **`use_cache = True`**: 训练时为了显存必须关闭 Cache，但**推理生成时必须开启**。开启后，模型在预测下一个 Token 时会重用之前的计算结果，推理速度会提升数倍。



---

## 💡 技术要点

### 显存与计算的艺术
通过这种方式微调 Gemma 4，你实际上是在进行一种**“外科手术式”的增强**：
* 原始的“知识库”（Base Model）被冷冻保存。
* 仅仅几百万个（相对于总参数量的极小部分）LoRA 参数在针对 `emotion` 数据集进行快速迭代。
* `train_result` 将返回训练期间的 **Loss（损失值）** 变化。通常情况下，你会看到 Loss 从一个较高的数值平滑下降，这代表模型正逐渐学会如何按照你的 System Prompt 准确给出情感标签。

---

### 🔎 观察输出
运行结束后，`train_result` 会展示最终的训练损耗、步数和总耗时。如果一切正常，你会看到一个百万级别的 `Trainable LoRA parameters` 数字，这证明了轻量化微调已经成功部署。

In [ ]:
from peft import PeftModel

if isinstance(base_model, PeftModel):
    base_model = base_model.unload()
    base_model.config.use_cache = False

trainer = SFTTrainer(
    model=base_model,
    train_dataset=sft_dataset["train"],
    eval_dataset=sft_dataset["validation"],
    peft_config=lora_config,
    args=training_args,
    processing_class=processor,
)

trainable_params = 0
for param in trainer.model.parameters():
    if param.requires_grad:
        trainable_params += param.numel()

if trainable_params == 0:
    raise RuntimeError("No trainable LoRA parameters were attached. Check target_modules before training.")

print(f"Trainable LoRA parameters: {trainable_params:,}")
train_result = trainer.train()
trainer.model.eval()
trainer.model.config.use_cache = True
train_result


Tokenizing train dataset:   0%|          | 0/4000 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/400 [00:00<?, ? examples/s]

Trainable LoRA parameters: 50,499,584


Step,Training Loss,Validation Loss
50,1.038701,0.309552
100,0.198841,0.192570
150,0.148183,0.136113
200,0.101560,0.122830
250,0.097333,0.094157


TrainOutput(global_step=250, training_loss=0.31692357444763186, metrics={'train_runtime': 756.1166, 'train_samples_per_second': 5.29, 'train_steps_per_second': 0.331, 'total_flos': 1.1993382889127424e+16, 'train_loss': 0.31692357444763186})

## 13. 保存适配器和处理器

这段代码执行的是微调流程的**持久化（Persistence）**操作。它的作用是将内存中辛苦训练出来的“学问”提取出来，保存为物理文件，以便将来可以在任何地方加载使用。

---

## 💾 模型与分词器持久化解析

### 1. `trainer.model.save_pretrained(...)`
* **保存内容**：此方法专门用于保存 **LoRA 适配器（Adapter）** 权重。
* **技术细节**：由于使用了 PEFT 技术，这里**并不会**保存整个 Gemma 4 的几十 GB 完整参数。它只保存：
    * **`adapter_model.bin`**：新训练出的轻量化层权重（通常只有几十 MB）。
    * **`adapter_config.json`**：LoRA 的配置信息（如秩 $r$、$\alpha$ 值、目标层等），用于告诉程序这些权重应该如何插回原始模型。
* **核心优势**：这种“分离式保存”极大节省了存储空间。你只需要保存这份“补丁”，以后加载时只需配合原版 Gemma 4 模型即可。

### 2. `processor.save_pretrained(...)`
* **保存内容**：保存 **Tokenizer（分词器）** 的所有配置和词表。
* **重要性**：在微调过程中，我们曾手动调整过 `pad_token` 和 `chat_template`。通过这一行，所有这些关于“如何将文本转为 ID”的自定义规则都会被封装进：
    * `tokenizer_config.json`
    * `special_tokens_map.json`
* **协同效应**：确保未来推理时使用的分词逻辑与训练时**完全对齐**。如果分词逻辑不一致，模型即便有最好的权重也会输出乱码。

### 3. 目录结构说明
运行后，`./gemma4-emotion-lora` 文件夹将包含一个完整的、可部署的微调成果。



---

## 💡 为什么这一步对 Gemma 4 微调至关重要？

* **断点续用**：Colab 的运行环境是临时的。一旦断开连接，内存中的模型就会消失。保存到本地（或挂载的 Google Drive）是保护训练成果的唯一方法。
* **解耦部署**：你可以将这个只有几百 MB 的文件夹上传到 Hugging Face Hub 或私人服务器。在部署时，只需调用 `PeftModel.from_pretrained(base_model, "./gemma4-emotion-lora")` 即可瞬间将一个通用的 Gemma 4 变成专业的情感分类专家。
* **版本管理**：你可以针对不同的超参数保存多个文件夹（如 `...-lora-v1`, `...-lora-v2`），方便进行横向的模型选优。

---

## 🔍 验证建议
运行完这段代码后，你可以点击 Colab 左侧的“文件”图标，确认目录下出现了 `adapter_model.bin` 和相关 JSON 文件。只要这些文件在，你的微调成果就是安全的。

In [ ]:
trainer.model.save_pretrained("./gemma4-emotion-lora")
processor.save_pretrained("./gemma4-emotion-lora")
print("Saved adapter and processor to ./gemma4-emotion-lora")

Saved adapter and processor to ./gemma4-emotion-lora


这段代码是微调流程的**最终交付阶段**。它的作用是将本地训练好的模型权重和配置文件上传至 **Hugging Face Hub** 云端仓库。这不仅是为了备份，更是为了实现模型的开源共享或远程部署。

---

## ☁️ 云端推送与模型托管解析

### 1. 仓库标识符 (`repo_id`)
* **内容**：`"用户名/仓库名"` 格式。
* **作用**：这决定了你的模型在 Hugging Face 平台上的唯一地址。上传成功后，任何人（或你自己在其他机器上）都可以通过这个 ID 直接加载你的微调成果。

### 2. 适配器推送 (`trainer.model.push_to_hub`)
* **推送内容**：与 `save_pretrained` 类似，它仅推送 LoRA 的层权重（`adapter_model.bin`）和配置文件（`adapter_config.json`）。
* **`private=False`**：设置仓库为**公开**。这意味着该模型将出现在你的 Hugging Face 个人主页上，社区用户可以查看并下载它。
* **自动化处理**：该函数会自动处理 Git 提交、LFS（大文件存储）上传，并生成一个初始的模型卡片（Model Card）。

### 3. 分词器推送 (`processor.push_to_hub`)
* **同步更新**：确保云端仓库不仅有“大脑”（权重），还有“嘴巴”（分词器）。
* **重要性**：由于微调时使用了特定的 Chat Template 和特殊 Token，将 `processor` 推送到同一个 `repo_id` 下，可以保证其他用户在调用 `AutoTokenizer.from_pretrained(repo_id)` 时，能直接获得与你完全一致的环境配置。

---

## 💡 技术细节与安全点

### A. 身份验证依赖
这段代码之所以能成功运行，前提是你已经在脚本开头执行了 `huggingface_hub.login` 并且使用了具有 **Write** 权限的 Token。如果没有登录，或者没有允许对应的权限，程序会在此处抛出权限错误。

### B. 增量上传
Hugging Face 的上传机制非常智能。如果你多次运行此代码，它会对比本地文件与云端文件的哈希值，仅上传发生变化的部分，极大节省了网络带宽。

### C. 远程加载预览
一旦推送完成，你就可以在任何支持 `peft` 的环境中，仅用几行代码恢复这个情感分类专家：
```python
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM

# 只需这两行即可从云端加载
base = AutoModelForCausalLM.from_pretrained("google/gemma-4-E4B-it")
model = PeftModel.from_pretrained(base, "kingabzpro/gemma4-emotion-lora")
```



---

## 🔍 执行反馈
最后一行 `print` 输出了一个可点击的链接。点击进入后，你可以在 Hugging Face 的 Web 界面上看到训练指标的图表（如果 Trainer 自动上传了 `runs` 文件夹），并直接在浏览器中测试你的微调模型。

In [ ]:
# Push adapter + processor to the Hub

repo_id = "kingabzpro/gemma4-emotion-lora"
trainer.model.push_to_hub(
    repo_id,
    private=False,
)

processor.push_to_hub(
    repo_id,
    private=False,
)

print(f"Pushed to https://huggingface.co/{repo_id}")


README.md: 0.00B [00:00, ?B/s]

HfHubHTTPError: (Request ID: Root=1-69ec7e24-3e32d63a2a3e6b124a1ab825;a45c432e-4a53-4b78-aede-f4614f79d9ee)

403 Forbidden: Authorization error..
Cannot access content at: https://huggingface.co/kingabzpro/gemma4-emotion-lora.git/info/lfs/objects/batch.
Make sure your token has the correct permissions.

## 14. 无需重新加载模型即可运行微调后评估


这段代码标志着微调实验的**最终收官阶段**。它不仅在逻辑上完成了模型状态的切换，还通过直接复用内存中的模型，巧妙地避开了 Colab 等受限环境下的显存陷阱，从而获取微调后的最终性能指标。

---

## 📈 微调后评估（Post-FT Evaluation）逻辑解析

### 1. 内存复用策略 (Memory Efficiency)
* **`ft_model = trainer.model`**：
    这是非常关键的实战技巧。在微调结束后，模型已经处于显存中。如果选择先删除模型再从硬盘重新加载，往往会因为 **显存碎片化（Memory Fragmentation）** 导致即使有空余显存也无法分配，最终触发 `OOM (Out of Memory)` 错误。直接从 `trainer` 对象中提取模型是最安全、最快速的做法。

### 2. 推理状态激活
* **`ft_model.eval()`**：
    确保模型进入**预测模式**。它会冻结 Dropout 层，保证对于相同的测试输入，模型的输出是稳定的。
* **`ft_model.config.use_cache = True`**：
    **这是推理加速的核心**。在训练期间（Forward Pass），为了节省梯度计算所需的显存，我们会关闭 KV Cache。但在评估阶段，开启 Cache 可以让模型在生成 Token 时复用之前的计算结果，使评估速度提升数倍。

### 3. 指标对撞 (Metrics Comparison)
* **`evaluate_model(ft_model, ...)`**：
    使用与微调前**完全一致**的评估函数和测试集。这种“变量控制法”确保了实验的严谨性。
* **`post_metrics` 预期变化**：
    运行此行后，你可以将其与之前的 `pre_metrics` 进行对比。理想情况下，你会观察到：
    * **准确率（Accuracy）**和 **F1 分数**大幅提升。
    * **`invalid_predictions` 降至接近 0**：这证明通过微调，Gemma 4 已经完美学会了“只返回一个标签”的指令格式。



---

## 💡 技术亮点：为什么这一步不可或缺？

### A. 验证微调的有效性
微调并不总是正向的。如果 `learning_rate` 过大或数据质量不佳，可能会发生“灾难性遗忘”。只有通过 `post_metrics`，你才能确定模型是在真正学习任务，还是在乱猜。

### B. 定位剩余问题
通过分析 `post_report`（详细报告），你可以看到哪些情感类别依然难以区分。例如，如果 `joy` 和 `love` 依然混淆，这能为你下一轮调整数据分布提供直接证据。

### C. 确定部署可行性
在将模型推向生产环境或 Hugging Face 之前，这份 `post_metrics` 就是你的模型“合格证”。它量化了模型在处理真实（未见过）文本时的可靠程度。

---

## 🔍 结果解读提示
当你看到 `post_metrics` 输出时，重点关注 **`macro_f1`**。如果它比微调前有了质的飞跃（例如从 0.4 提升到 0.9），说明你的 Gemma 4 已经成功转型为一名精准的情感分类专家。

In [ ]:
# Reuse the in-memory fine-tuned model to avoid a second base-model load.
# On smaller GPUs, reloading after training often causes fragmentation or OOMs.
ft_model = trainer.model
ft_model.eval()
ft_model.config.use_cache = True
post_metrics, post_report, post_preds = evaluate_model(ft_model, processor, "test")
post_metrics


Evaluating test:   0%|          | 0/400 [00:00<?, ?it/s]

{'accuracy': 0.92,
 'macro_f1': 0.862603224311464,
 'invalid_predictions': 0,
 'evaluated_examples': 400}

这段代码的作用是将微调后生成的详细评估报告转换为一个**结构化的表格**。通过这种可视化方式，你可以清晰地对比微调前后的性能差异，并精准识别模型在特定情感类别上的表现。

---

## 📊 微调后评估报告表格化解析

### 1. 数据结构转换逻辑
* **`pd.DataFrame(post_report)`**:
    `post_report` 是由 `classification_report` 函数生成的嵌套字典。将其转换为 DataFrame 后，默认会将类别名称（如 `sadness`, `joy` 等）作为列，而将指标（精确率、召回率等）作为行。
* **`.transpose()` (或 `.T`)**:
    执行**矩阵转置**。将行与列对调，使每一行代表一个具体的情感类别，每一列代表一个评估指标。这种布局符合数据科学中分析分类结果的标准习惯，极大方便了横向对比。

### 2. 核心评估指标解析

转置后的表格会包含以下关键列，它们共同构成了微调效果的“成绩单”：

| 指标名称 | 定义与意义 | 微调后的预期表现 |
| :--- | :--- | :--- |
| **`precision` (精确率)** | 模型预测为该情感的样本中，实际也为该情感的比例。 | 反映了模型分类的**准确度**。高精确率意味着模型很少“误报”。 |
| **`recall` (召回率)** | 实际为该情感的样本中，被模型正确找出来的比例。 | 反映了模型分类的**查全率**。高召回率意味着模型很少“漏报”。 |
| **`f1-score`** | 精确率和召回率的调和平均数。 | **最重要的综合指标**。它能平衡精确率和召回率，最能体现模型在复杂样本上的真实战斗力。 |
| **`support`** | 测试集中该类别的真实样本数量。 | 提醒你数据的分布。如果某个类别的 `support` 很少，其指标的波动可能具有偶然性。 |



---

## 💡 微调 Gemma 4 的成效验证

通过观察这个“微调后”的表格，你可以获得以下核心结论：

* **性能飞跃对比**：将此表与微调前的 `pre_report` 对比，你会发现 F1 分数通常会有显著提升（例如从 0.3-0.5 提升到 0.8-0.9 以上）。这证明了 **LoRA 适配器**成功捕捉到了 `emotion` 数据集的语义特征。
* **特定类别的攻克**：在微调前，Gemma 4 可能难以区分 `love` 和 `joy`。微调后的报表能告诉你模型是否已经学会了这两个类别之间的细微边界。
* **平均分（Macro/Weighted Avg）**：
    * **`macro avg`**：展示了模型在所有类别（无论样本多少）上的平均水平。
    * **`weighted avg`**：根据样本量加权后的水平。
    通常在微调后，这两项数值会趋于接近且处于高位。

---

## 🔍 技术总结
这一步不仅是数据的展示，更是**微调闭环的最后验证**。它将模型从“黑盒”生成的文字变成了可量化、可审计的科学数据，确保你对微调后的 Gemma 4 性能心中有数。

In [ ]:
pd.DataFrame(post_report).transpose()

,precision,recall,f1-score,support
sadness,0.939130,0.947368,0.943231,114.00
joy,0.918605,0.975309,0.946108,162.00
love,0.857143,0.692308,0.765957,26.00
anger,0.976190,0.872340,0.921348,47.00
fear,0.871795,0.944444,0.906667,36.00
surprise,0.818182,0.600000,0.692308,15.00
accuracy,0.920000,0.920000,0.920000,0.92
macro avg,0.896841,0.838628,0.862603,400.00
weighted avg,0.919247,0.920000,0.917602,400.00


这段代码生成的**微调后混淆矩阵**是整个实验的“最终判决书”。通过它，你可以直观地看到经过 **LoRA** 微调后的 Gemma 4 在各个情感类别之间的辨析能力，以及它在遵守指令方面的进化。

---

## 🏁 微调后混淆矩阵深度解析

### 1. 矩阵结构的对比意义
与微调前的矩阵相比，这个版本的表格通常会呈现出明显的**“对角线集中”**现象：
* **主对角线（自左上至右下）**：单元格中的数值应该显著增大。这代表大部分测试样本都被准确地分类到了对应的标签中。
* **非对角线区域**：数值应该大幅减少，趋近于 0。这说明模型原本存在的认知模糊（如混淆“生气”与“恐惧”）已通过微调得到了有效修复。

### 2. `INVALID` 列的消失
这是微调效果最直观的体现：
* 在微调前，模型可能会因为输出长句或非预定义词汇导致出现 `INVALID` 预测。
* **微调后**：如果该列几乎全为 0，说明模型已经彻底掌握了你在 `SYSTEM_PROMPT` 中要求的“仅输出标签”的格式规范。这证明了 **SFT（有监督微调）** 在对齐（Alignment）方面的强大威力。



---

## 💡 关键诊断点

通过观察这个矩阵，你可以进行深度的错误分析：

* **语义近义词挑战**：观察 `joy` 与 `love` 之间的交叉点。即使是微调后的模型，也可能在这两类上存在轻微混淆。如果数值依然较高，说明这两个类别的特征在当前数据集中非常接近。
* **样本均衡性影响**：参考之前 `post_report` 中的 `support` 值。如果某个样本量很少的类别（如 `surprise`）在对角线上数值较低，说明模型对冷门类别的学习尚不充分。
* **零错判目标**：对于情感极性截然不同的类别（如 `joy` vs `sadness`），理论上微调后的混淆矩阵在这些交叉点上应为 0。如果仍有数值，则需要检查训练数据中是否存在标签错误（Label Noise）。

---

## 🔍 技术总结

这份混淆矩阵不仅仅是数字的堆砌，它是**模型认知的地图**。
* **主对角线越“亮”**：模型的分类边界越清晰。
* **非对角线越“暗”**：模型的鲁棒性和确定性越高。

至此，你已经完成了从环境配置、数据工程、模型量化加载、LoRA 适配器训练到最终量化评估的全流程。这份矩阵为你微调 Gemma 4 的实验画上了一个完美的句号。

In [ ]:
confusion_matrix_df(post_preds)


,sadness,joy,love,anger,fear,surprise,INVALID
sadness,108,3,0,1,2,0,0
joy,1,158,3,0,0,0,0
love,1,7,18,0,0,0,0
anger,4,1,0,41,1,0,0
fear,0,0,0,0,34,2,0
surprise,1,3,0,0,2,9,0
INVALID,0,0,0,0,0,0,0


## 15. 对比微调前后效果

这段代码的作用是将微调前（Baseline）和微调后（Post-FT）的量化指标整合进一个单一的 **Pandas DataFrame** 中。这是整个实验的“成绩单汇总”，通过直观的横向对比，科学地证明微调对模型性能的提升效果。

---

## 📊 性能对比汇总表解析

### 1. 字典拆解与合并逻辑 (`**` 展开符)
* **`**pre_metrics` 和 `**post_metrics`**:
    这是 Python 的字典拆解语法（Dictionary Unpacking）。它将之前 `evaluate_model` 函数返回的字典内容（如 `accuracy`, `macro_f1` 等）平铺开来。
* **结构化构建**:
    代码通过列表嵌套字典的方式，为每一组指标手动添加了一个名为 `stage` 的标签列。最终生成的表格结构如下：
    
| stage | accuracy | macro_f1 | invalid_predictions | evaluated_examples |
| :--- | :--- | :--- | :--- | :--- |
| **pre_finetuning** | $0.xx$ | $0.xx$ | $N$ | 400 |
| **post_finetuning** | $0.yy$ | $0.yy$ | $M$ | 400 |

### 2. 核心对比维度解析

通过这个表格，你可以从以下三个关键维度评估 Gemma 4 的进化：

#### A. 任务准确度提升 ($\Delta Accuracy$)
* **意义**: 展示了模型在分类任务上的绝对正确率增长。
* **预期**: 微调后的 `accuracy` 通常会有大幅度的飞跃，证明 LoRA 适配器成功让模型掌握了 `dair-ai/emotion` 数据集的特定分布。

#### B. 类别平衡表现 ($\Delta Macro\_F1$)
* **意义**: 相比准确率，`macro_f1` 考虑了每一个情感类别的贡献。
* **深度价值**: 如果微调前后的 $Macro\_F1$ 提升比例高于 $Accuracy$，说明微调不仅提升了整体性能，还显著改善了模型对那些“难样本”或“小众类别”（如 `surprise`）的识别能力。

#### C. 指令遵循能力 ($\Delta Invalid\_Predictions$)
* **意义**: 这是衡量生成式模型（LLM）做分类任务是否“听话”的最直接指标。
* **关键点**: 微调前，Gemma 4 可能会输出多余的解释；微调后，该数值应趋近于 $0$。这代表模型已经从一个“话痨助手”被训练成了一个“精准分类器”。



---

## 💡 在微调 Gemma 4 流程中的战略价值

### 验证“对齐”效果
微调不仅仅是让模型变聪明，更是为了让模型**与你的具体指令（System Prompt）对齐**。这个汇总表用数据证明了模型是否已经内化了你定义的规则。

### 汇报与记录
在实际的项目开发中，这个 `comparison_df` 是最适合放入实验报告或演示文稿（PPT）的内容。它将复杂的深度学习训练过程浓缩为几组具有强说服力的对比数据。

### 决策支持
如果对比发现 `post_finetuning` 的提升不明显，或者 `invalid_predictions` 依然很高，这将直接提示开发者需要检查训练数据的质量、调整 LoRA 的 $r$ 值，或者增加训练的 Epochs。

---

## 🔍 技术总结
这一行代码完成了从“个体数据点”到“系统性结论”的升华。它将原本分散在两个阶段的实验结果强制对齐，为整个微调工程提供了一个清晰、量化的终点。

In [ ]:
comparison_df = pd.DataFrame([
    {"stage": "pre_finetuning", **pre_metrics},
    {"stage": "post_finetuning", **post_metrics},
])
comparison_df


,stage,accuracy,macro_f1,invalid_predictions,evaluated_examples
0,pre_finetuning,0.575,0.417038,38,400
1,post_finetuning,0.920,0.862603,0,400


这段代码执行的是微调效果的**样本级对比分析（Case Study）**。通过将微调前后的预测结果横向合并，并筛选出那些“模型改变了主意”的样本，你可以深入观察微调是如何具体改变模型认知逻辑的。

---

## 🔍 预测变更对比分析解析

### 1. 数据对齐与合并逻辑
* **`pre_preds.copy()`**: 创建微调前预测结果的副本，确保后续操作不会修改原始评估数据。
* **列名重构 (`rename`)**: 将原始的 `pred_label` 和 `correct` 改名为 `pre_pred` 和 `pre_correct`。这是为了在合并后能够清晰地区分哪些是“微调前”的表现。
* **特征注入**: 直接将 `post_preds`（微调后）的预测结果和正确性判定添加为新列。由于 `pre_preds` 和 `post_preds` 是对同一组测试集按相同顺序评估的，这种按索引对齐的方式非常高效且准确。

### 2. 差异化筛选 (`pre_pred != post_pred`)
这是代码的灵魂所在。它过滤掉了模型在微调前后看法一致的样本，只保留了**预测发生变化**的行。
* **捕捉进化轨迹**: 这里的样本包含了模型从“错”变“对”的成功案例，也可能包含从“对”变“错”的过拟合案例。
* **排除干扰**: 那些微调前后都预测正确的简单样本（Easy Examples）被过滤掉，让你能集中精力分析那些位于分类边界（Decision Boundary）上的复杂样本。

### 3. `head(20)` 的直观观察
展示前 20 条变化样本，能让你在不进行大规模统计的情况下，获得直观的“体感”反馈：
* **指令遵循的改变**: 你可能会看到 `pre_pred` 中有很多被标记为 `INVALID` 的项，而在 `post_pred` 中它们变成了具体的标签（如 `joy`）。
* **语义认知的修正**: 例如，某个描述“被背叛”的句子，微调前被误判为 `anger`，微调后被准确识别为 `sadness`（这通常取决于数据集 `dair-ai/emotion` 的具体定义）。

---

## 💡 在微调 Gemma 4 流程中的诊断价值

### A. 评估“对齐”的广度
如果大部分变化的样本都伴随着 `pre_correct = False` 到 `post_correct = True` 的转变，说明微调在正面修复模型的认知。

### B. 发现数据集偏好
大模型在微调后往往会表现出数据集特有的“偏见”。通过观察这些变化的样本，你可以发现 `dair-ai/emotion` 这个数据集是如何定义细微情感边界的，以及 Gemma 4 是否成功接收到了这些信号。

### C. 识别过拟合（Overfitting）风险
如果你发现某些原本在微调前预测正确的样本，在微调后反而预测错了（`pre_correct=True` -> `post_correct=False`），这通常是一个强烈的信号，提示你可能训练轮数（Epochs）过多，或者 LoRA 的秩（Rank）设置得过高，导致模型在丢弃通用常识来迎合训练数据的噪声。

---

## 🔍 结果预览说明
执行此代码后，你将看到一个包含 `text`, `true_label`, `pre_pred`, `post_pred` 等列的表格。
* **重点关注**：`pre_correct` 为 `False` 且 `post_correct` 为 `True` 的行，它们是你微调成功的勋章。
* **深度分析**：阅读那些模型依然预测错误的文本，思考是否需要通过补充数据或优化 Prompt 来进一步改进。

In [ ]:
merged_examples = pre_preds.copy()
merged_examples = merged_examples.rename(columns={"pred_label": "pre_pred", "correct": "pre_correct"})
merged_examples["post_pred"] = post_preds["pred_label"]
merged_examples["post_correct"] = post_preds["correct"]

changed_predictions = merged_examples[merged_examples["pre_pred"] != merged_examples["post_pred"]]
changed_predictions.head(20)


,text,true_label,pre_pred,raw_pred_label,pre_correct,post_pred,post_correct
2,i feel is that the most likeable characters ar...,joy,sadness,sadness,False,joy,True
5,im feeling and if ive liked being pregnant,love,joy,joy,False,love,True
7,i used to be able to hang around talk with the...,anger,sadness,sadness,False,anger,True
8,i don t have the feeling of divine vibrations,joy,sadness,sadness,False,joy,True
9,i vented my feelings towards the pathetic excu...,sadness,anger,anger,False,sadness,True
11,i get the feeling that this could be dangerous,anger,fear,fear,False,anger,True
12,i feel like some of you have pains and you can...,joy,sadness,sadness,False,joy,True
13,i cant write a review for a book i adore unles...,love,joy,joy,False,love,True
14,i mentioned in my last blog that i have starte...,fear,sadness,sadness,False,fear,True
16,i was wondering if you will focus on the probl...,sadness,INVALID,angerequ,False,sadness,True


## 16. 使用内存中的微调后模型进行预测


这段代码定义并运行了一个**面向终端用户**的推理函数，它专门调用了你刚刚微调完成的 **Gemma 4 (FT)** 模型。这标志着模型已经从“训练实验室”正式走入“实际应用”场景。

---

## 🛠️ 微调后推理函数 `predict_emotion_ft` 解析

### 1. 专用模型绑定
* **`ft_model`**: 与之前的 `predict_emotion` 不同，这个函数强制指定使用 `ft_model`（即加载了 LoRA 适配器后的模型）。
* **逻辑隔离**: 这种写法确保了你是在调用“进化后”的专家模型，而不是原始的通用模型。它能直接利用微调过程中学到的情感分类边界和指令遵循规范。

### 2. 简化的调用接口
* **封装性**: 函数隐藏了底层复杂的推理细节（如 Token 转换、设备映射、解码截断和正则清洗）。对于最终用户或上游业务逻辑来说，只需要输入一段文本，就能直接得到一个标准的情感标签字符串。
* **参数一致性**: 它自动注入了 `processor` 和训练时完全一致的 `SYSTEM_PROMPT`。这种**训练与推理的镜像对称**是保证大模型在生产环境下表现稳定的基石。

### 3. 推理过程演示：以 "I feel completely heartbroken and alone." 为例
当你运行这一行时，后台的完整链路如下：

* **输入封装**: 将文本包装成 `<start_of_turn>user...heartbroken...<end_of_turn>` 等 Gemma 特有的对话格式。
* **语义理解**: `ft_model` 接收输入。由于它在 `dair-ai/emotion` 数据集上进行了微调，其权重对 `heartbroken`（心碎）和 `alone`（孤独）等词汇具有极高的情感敏感度。
* **精准生成**: 模型不会输出“我很遗憾听到这个消息”，而是受过训地直接锁定最匹配的标签 Token（例如：`sadness`）。
* **结果归一化**: 即使模型输出了带有空格或标点的结果，外层的 `generate_label` 也会通过正则将其清洗为纯净的标签。

---

## 💡 技术价值

### 验证“场景迁移”能力
通过手动输入像 "heartbroken" 这样充满强烈情感但可能不在训练集原句中的文本，你可以测试微调后模型的**泛化能力**。一个成功的微调模型应该能理解同义词和复杂的语境表达。

### 部署准备
这个函数实际上就是未来 API 接口的原型。如果你要将这个情感分类功能部署到服务器上，这个函数的核心逻辑将被直接迁移到推理框架（如 vLLM 或 TGI）中。

### 指令遵循的最终确认
对于 "heartbroken" 这种容易引发模型“安慰倾向”的词汇，通用模型往往会给出长篇大论。而运行 `predict_emotion_ft` 如果能干脆利落地只返回 `sadness`，则证明你的 **SFT（有监督微调）** 彻底重塑了模型的输出偏好。

In [ ]:
def predict_emotion_ft(user_text: str) -> str:
    return generate_label(ft_model, processor, user_text, SYSTEM_PROMPT)

predict_emotion_ft("I feel completely heartbroken and alone.")


'sadness'

这行代码是对微调后的模型进行的一次**实时性能验证**。通过输入一段带有极端正面情绪的文本，你可以直观地确认模型是否已经从通用的对话模式切换到了精准的分类模式。

---

## 🧪 实时推理与分类验证解析

### 1. 语义特征识别
* **文本内容**：`"This is the best day of my life!"`
* **情感信号**：包含 `"best day"` 和 `"life"` 这种具有强烈积极导向的关键词。
* **模型响应**：在 `dair-ai/emotion` 数据集的语境下，这种表达通常对应 **`joy` (喜悦)** 类别。经过微调的 **Gemma 4** 能够迅速捕捉到这种语调，并抑制生成其他描述性文字的倾向。

### 2. 微调后的行为特征
与微调前的原始模型相比，调用 `predict_emotion_ft` 会表现出以下显著改进：

* **极高的决策确定性**：由于模型在 4000 条情感数据上进行了 SFT（有监督微调），它对“什么是喜悦”有了更明确的数学定义，不再会在 `joy` 和 `love` 之间犹豫不决。
* **严格的格式遵循**：原始模型面对这类句子可能会回答 `"I'm so glad to hear that! It sounds like you are feeling joy."`，但微调后的模型会直接输出单词 **`joy`**，完美符合你定义的系统约束。

### 3. 函数调用链复盘
当你运行这一行时，代码内部经历了最后的完整循环：
1. **注入模板**：将文本嵌入到包含 `SYSTEM_PROMPT` 的对话结构中。
2. **加速推理**：`ft_model` 利用加载的 **LoRA 权重** 和开启的 **KV Cache**，在几毫秒内完成计算。
3. **后处理清洗**：通过 `generate_label` 中的正则逻辑，确保哪怕模型输出带有一个换行符，最终返回给你的也是干净的字符串。

---

## 💡 技术总结

这个简单的调用实际上是整个微调工程的**终点站**。
* 如果输出 **`joy`**：证明模型在**语义理解**和**指令遵循**两个维度都达到了预期目标。
* **应用价值**：这证明了你的模型已经具备了处理真实世界非结构化文本的能力，可以正式投入到情感监测、社交媒体分析或自动化客服等实际生产场景中。

In [ ]:
predict_emotion_ft("This is the best day of my life!")

'joy'

## 17. 可选：保存对比结果

这段代码标志着整个数据科学流程的**数据持久化（Exporting）**环节。它将内存中的实验结论和详细分析结果转化为标准化的 CSV 文件，确保你的微调成果可以被用于后续的报告编写、数据可视化或进一步的离群值分析。

---

## 💾 数据导出与存储逻辑解析

### 1. `comparison_df.to_csv(...)`
* **保存内容**：保存微调前后的核心性能指标对比（如 Accuracy, F1-score 等）。
* **技术价值**：这是实验的“总结性报告”。通过 `index=False` 参数，避免了将 Pandas 的默认行索引保存为一列无用的数据，使文件更加整洁。该文件通常用于在 Excel、Tableau 或 PowerBI 中创建性能提升柱状图。

### 2. `merged_examples.to_csv(...)`
* **保存内容**：保存每一条测试样本的详细预测记录，包含：
    * 原始文本 (`text`)
    * 真实标签 (`true_label`)
    * 微调前预测 (`pre_pred`)
    * 微调后预测 (`post_pred`)
    * 是否正确 (`pre_correct`, `post_correct`)
* **深度分析价值**：这是进行**错误分析（Error Analysis）**的核心素材。你可以通过筛选 `post_correct == False` 的行，在本地编辑器中仔细阅读模型依然无法正确分类的文本。这种离线分析对于发现训练数据的局限性至关重要。

### 3. 文件命名规范
* 使用了语义清晰的文件名（`before_after_metrics` 和 `prediction_examples`），方便在复杂的项目文件夹中快速定位数据，无需重新运行代码块即可随时回顾实验细节。

---

## 💡 在微调 Gemma 4 流程中的意义

### A. 突破 Colab 环境限制
由于 Google Colab 的磁盘空间是临时的，所有未导出的数据在会话结束（Disconnect）后都会丢失。通过执行这两行代码，你将数据写入了当前的虚拟机磁盘。

### B. 团队协作与审计
在企业级 AI 开发中，直接提供 Python 代码通常是不够的。导出的 CSV 文件可以作为“实验证据”提交给其他同事或团队，让他们在不运行任何代码的情况下，验证你的 Gemma 4 微调效果是否达标。

### C. 自动化流水线
如果你正在构建一个自动化的微调流水线，这些导出的 CSV 文件可以作为后续自动化测试脚本（CI/CD）的输入，用于判断新训练的模型是否优于旧版本，从而决定是否上线。

---

## 🔍 结语
到此为止，你已经完成了从环境搭建、数据工程、模型量化微调、云端推送、到最终数据分析并导出的**全生命周期微调实验**。这些 CSV 文件是你此次 Gemma 4 微调工程最坚实的量化资产。

In [ ]:
comparison_df.to_csv("gemma4_emotion_before_after_metrics.csv", index=False)
merged_examples.to_csv("gemma4_emotion_prediction_examples.csv", index=False)
print("Saved CSV outputs.")

Saved CSV outputs.


这次 Code Lab 完整地演示了如何利用 **PEFT (LoRA)** 和 **bitsandbytes (4-bit 量化)** 技术，在受限的硬件环境下对 **Gemma 4** 这种超大规模语言模型进行高效的专项任务微调。



总结来看，我们通过以下核心步骤实现了从通用模型到专用专家的转变：
1.  **环境与安全：** 建立了基于 Hugging Face 令牌的安全认证体系。
2.  **数据工程：** 完成了从原始数据集加载、随机抽样到符合 ChatML 标准的指令格式化转化。
3.  **量化加载：** 运用 NF4 量化与 BF16 计算精度，在大幅降低显存占用的同时保持了模型性能。
4.  **轻量化微调：** 仅训练不到 1% 的参数（LoRA），精准调整了模型在特定情感领域的决策边界。
5.  **科学评估：** 构建了从混淆矩阵到 F1 分数的全方位对比体系，不仅验证了准确率的飞跃，更量化证明了模型在指令遵循（减少 Invalid 输出）上的显著进化。

最终导出的适配器权重与评估报告，标志着该模型已具备从实验环境走向实际生产场景的成熟度。